# Conversational reasoning, with no training and no pretrained weights

Built from commit `535d024`, pre-registered in `PREREGISTRATION_convo.md`.

A dialogue states only adjacent pairs, shuffled, some by pronoun, 10% of them backwards, with
distractor turns in between. Then it asks about pairs that were never put side by side.

The brain sees each sentence once and is never trained: words become sparse codes on dendritic branches,
prefrontal working memory holds who is being talked about, a hippocampal map places each fact one-shot, an
episodic store keeps every utterance, and a basal-ganglia gate learns which of them answers. Conduction delays
decide what has arrived by the time the gate must choose.

The transformer and GRU are trained here on 20,000 batches of dialogues of exactly this kind. That is what the
GPU is for - the controls, not the brain.

In [ ]:
import os, torch
os.makedirs("/kaggle/working/neuromoe/tests", exist_ok=True); os.makedirs("/kaggle/working/neuromoe/chat", exist_ok=True)
os.makedirs("/kaggle/working/neuromoe/brain", exist_ok=True); os.makedirs("/kaggle/working/results", exist_ok=True)
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
%%writefile /kaggle/working/neuromoe/neuromoe.py
"""NeuroMoE: a growing, gradient-free sparse mixture of experts.

No backprop and no pretrained weights anywhere. The model starts empty and grows as it is used:

  SparseExpansion  fixed random sparse wiring + k-winners-take-all (fly mushroom body / dentate gyrus). Similar
                   inputs share active units, different inputs barely overlap. Never trained.
  ExpertPool       experts are assemblies with input synapses (a prototype over the sparse code) and output synapses.
                   Routing is lateral inhibition: the best-matching experts win. When nothing matches well enough a
                   new expert is born (neurogenesis) wired to exactly the active units (synaptogenesis); when the
                   winner predicts wrong, the match threshold is raised above it and the search continues (ARTMAP
                   match tracking), so exceptions get their own expert instead of overwriting old ones. Winners drift
                   toward their inputs (Hebbian instar: grow synapses to active units, weaken the rest), weak synapses
                   are pruned, experts that stop winning die.
  WorkingMemory    one-shot associative store written by an error-correcting local rule
                   (M += beta (v - M k) k^T / |k|): the value is present when it is written, so no gradient is needed.
                   Sparse keys are nearly orthogonal, so capacity scales with the code size, not with training. A
                   fixed "time cell" code per step lets it bind items to positions (serial recall, n-back).

Every update is local: it uses only the activity on either side of a synapse plus a target that is physically
present at that moment (a label, the next symbol, a stored value). Nothing is differentiated.
"""
import math

import torch


def _generator(seed):
    return torch.Generator().manual_seed(int(seed))


class SparseExpansion:
    """d_in -> n units, each summing `fan_in` randomly chosen inputs with fixed positive weights; exactly k fire.
    Positive, distinct weights make the code deterministic (no ties). An optional homeostatic bias equalises how often
    each unit fires, so no unit is dead - a local rule on each unit's own firing rate."""

    def __init__(self, d_in, n=2048, k=32, fan_in=8, seed=0, homeostasis=0.0):
        g = _generator(seed)
        self.d_in, self.n, self.k = d_in, n, k
        # every input must reach at least 2k units, or a lone active input (a symbol) cannot fill the k winners and
        # the rest are ties at zero - the same filler units for every symbol, which makes keys collide
        fan_in = max(fan_in, math.ceil(2 * k * d_in / n))
        # connections spread evenly over inputs, then shuffled: no input is under-represented by chance
        wiring = torch.arange(n * fan_in) % d_in
        self.idx = wiring[torch.randperm(n * fan_in, generator=g)].reshape(n, fan_in)
        self.w = 0.5 + torch.rand(n, fan_in, generator=g)
        self.bias = torch.zeros(n)
        self.rate = torch.full((n,), k / n)
        self.homeostasis = homeostasis

    @torch.no_grad()
    def __call__(self, x):
        """x (B, d_in) -> binary code (B, n) with exactly k ones."""
        drive = (x[:, self.idx] * self.w).sum(-1) - self.bias
        code = torch.zeros(x.shape[0], self.n)
        code.scatter_(1, drive.topk(self.k, dim=1).indices, 1.0)
        if self.homeostasis:
            self.rate.mul_(0.99).add_(0.01 * code.mean(0))
            self.bias.add_(self.homeostasis * (self.rate - self.k / self.n))
        return code


class ExpertPool:
    """Neurogenesis mixture of experts over binary sparse codes.

    match(e, x) = (P_e . x) / |x| in [0, 1]: the share of the input's active units that expert e is wired to.
    labels="exact": classification; ARTMAP match tracking recruits a new expert whenever the accepted one predicts
    the wrong label. labels="stochastic": next-symbol prediction; an expert splits (a new, more specific expert is
    born) only when it has seen enough evidence and still gave the observed symbol low probability - so contexts are
    refined exactly where the data demands it (variable-order memory)."""

    def __init__(self, n_code, n_out, vigilance=0.7, k_route=3, temperature=0.05, beta=0.2, prune=0.05,
                 max_experts=4096, labels="exact", split_prob=0.1, split_evidence=4, smoothing=0.1, trust=0.0):
        self.n_code, self.n_out = n_code, n_out
        self.vigilance, self.k_route, self.temperature = vigilance, k_route, temperature
        self.beta, self.prune, self.max_experts = beta, prune, max_experts
        self.labels, self.split_prob, self.split_evidence, self.smoothing = labels, split_prob, split_evidence, smoothing
        # trust > 0: an expert's vote is scaled by evidence / (evidence + trust), so a newborn expert resting on one
        # example (possibly a mislabelled one) is heard weakly until other examples confirm it
        self.trust = trust
        self.P = torch.zeros(0, n_code)  # input synapses (prototype strengths)
        self.O = torch.zeros(0, n_out)   # output synapses (association counts)
        self.last_win = torch.zeros(0)
        self.step = 0
        self.born = self.died = 0

    def __len__(self):
        return self.P.shape[0]

    @torch.no_grad()
    def match(self, code):
        return self.P @ code / code.sum().clamp_min(1)

    @torch.no_grad()
    def predict(self, code, return_match=False):
        """Distribution over outputs: the top k_route experts vote, weighted by match (lateral inhibition)."""
        if not len(self):
            p = torch.full((self.n_out,), 1.0 / self.n_out)
            return (p, 0.0) if return_match else p
        m = self.match(code)
        top = m.topk(min(self.k_route, len(self))).indices
        w = self._weights(m[top], top)
        votes = self.O[top] + self.smoothing
        p = (w[:, None] * votes / votes.sum(1, keepdim=True)).sum(0)
        return (p, m.max().item()) if return_match else p

    def _weights(self, match, idx):
        w = torch.softmax(match / self.temperature, -1)
        if self.trust:
            evidence = self.O[idx].sum(-1)
            w = w * evidence / (evidence + self.trust)
            w = w / w.sum(-1, keepdim=True).clamp_min(1e-12)
        return w

    @torch.no_grad()
    def predict_batch(self, codes):
        """Vectorised predict for many inputs at once: (distributions (B, n_out), best match (B,))."""
        B = codes.shape[0]
        if not len(self):
            return torch.full((B, self.n_out), 1.0 / self.n_out), torch.zeros(B)
        m = codes @ self.P.T / codes.sum(1, keepdim=True).clamp_min(1)
        top = m.topk(min(self.k_route, len(self)), dim=1)
        w = self._weights(top.values, top.indices)
        votes = self.O[top.indices] + self.smoothing
        return (w[..., None] * votes / votes.sum(-1, keepdim=True)).sum(1), m.max(1).values

    @torch.no_grad()
    def _birth(self, code, y):
        if len(self) >= self.max_experts:  # full: the longest-idle expert dies first
            self._kill(self.last_win.argmin())
        self.P = torch.cat([self.P, code[None]])
        o = torch.zeros(1, self.n_out)
        o[0, y] = 1.0
        self.O = torch.cat([self.O, o])
        self.last_win = torch.cat([self.last_win, torch.tensor([float(self.step)])])
        self.born += 1
        return len(self) - 1

    @torch.no_grad()
    def _kill(self, e):
        keep = torch.ones(len(self), dtype=torch.bool)
        keep[e] = False
        self.P, self.O, self.last_win = self.P[keep], self.O[keep], self.last_win[keep]
        self.died += 1

    @torch.no_grad()
    def _reinforce(self, e, code, y):
        """Hebbian instar with synaptic turnover: grow toward active units, weaken the rest, prune weak synapses."""
        self.P[e] += self.beta * (code - self.P[e])
        self.P[e][self.P[e] < self.prune] = 0.0
        self.O[e, y] += 1.0
        self.last_win[e] = float(self.step)

    @torch.no_grad()
    def learn(self, code, y):
        """One labelled example, seen once. Returns the expert that learned it."""
        self.step += 1
        if not len(self):
            return self._birth(code, y)
        m = self.match(code)
        rho = self.vigilance
        for e in m.argsort(descending=True).tolist():
            if m[e] < rho:
                break  # nothing left clears the (possibly raised) vigilance
            if self.labels == "exact":
                if self.O[e].argmax().item() == y:
                    self._reinforce(e, code, y)
                    return e
                rho = m[e].item() + 1e-6  # match tracking: this expert is wrong here, demand a closer one
                continue
            # stochastic: accept the best match unless it has confidently learned that y is unlikely here
            counts = self.O[e]
            p_y = (counts[y] + self.smoothing) / (counts.sum() + self.smoothing * self.n_out)
            if counts.sum() >= self.split_evidence and p_y < self.split_prob and m[e] < 0.999:
                break  # a surprise in a context it knows well: split, give this context its own expert
            self._reinforce(e, code, y)
            return e
        return self._birth(code, y)

    def synapses(self):
        return int((self.P > 0).sum()) + int((self.O > 0).sum())


class WorkingMemory:
    """One-shot associative store: M (n_value x n_key), written M += beta (v - M k) k^T / |k|. After a write with
    beta = 1 the key recalls its value exactly; earlier bindings are disturbed only through shared active units.
    `decay` < 1 lets old items fade (a leaky buffer); 1 keeps everything until overwritten."""

    def __init__(self, n_key, n_value, beta=1.0, decay=1.0):
        self.M = torch.zeros(n_value, n_key)
        self.beta, self.decay = beta, decay

    @torch.no_grad()
    def write(self, key, value):
        if self.decay < 1.0:
            self.M.mul_(self.decay)
        self.M += self.beta * torch.outer(value - self.M @ key, key) / key.sum().clamp_min(1)

    @torch.no_grad()
    def read(self, key):
        return self.M @ key

    def clear(self):
        self.M.zero_()

    def synapses(self):
        return int((self.M != 0).sum())


class TimeCells:
    """A fixed sparse code for every time step (hippocampal time cells): step t has its own random k-subset of n
    units, independent of content, so items can be bound to *when* they arrived and replayed in order."""

    def __init__(self, n=2048, k=32, seed=1, max_steps=4096):
        g = _generator(seed)
        self.codes = torch.zeros(max_steps, n)
        for t in range(max_steps):
            self.codes[t, torch.randperm(n, generator=g)[:k]] = 1.0

    def __call__(self, t):
        return self.codes[t]


class NeuroMoE:
    """The assembled model for a symbolic or vector stream: expansion (+ optional context trace) -> expert pool,
    with a working memory alongside. `context` > 0 feeds a decaying trace of recent inputs into the expansion, so the
    same symbol after different histories lands on different experts."""

    def __init__(self, d_in, n_out, n=2048, k=32, fan_in=8, context=0.0, context_decay=0.5, seed=0, **pool):
        self.d_in, self.context, self.context_decay = d_in, context, context_decay
        self.expand = SparseExpansion(d_in * (2 if context else 1), n=n, k=k, fan_in=fan_in, seed=seed)
        self.pool = ExpertPool(n, n_out, **pool)
        self.trace = torch.zeros(d_in)

    @torch.no_grad()
    def encode(self, x):
        """x (d_in,) -> sparse code (n,). With context on, the current input is concatenated with the trace."""
        inp = torch.cat([x, self.context * self.trace]) if self.context else x
        return self.expand(inp[None])[0]

    @torch.no_grad()
    def advance(self, x):
        if self.context:
            self.trace.mul_(self.context_decay).add_(x)

    def reset(self):
        self.trace.zero_()

    def footprint(self):
        """Stored synapses (nonzero weights) - the model's actual size, which grows with use."""
        return self.pool.synapses()

In [ ]:
%%writefile /kaggle/working/neuromoe/cogmap.py
"""CognitiveMap: pseudospatial memory. Items are placed in a low-dimensional latent space by their *relations*, not by
how they look, so geometry carries meaning and relations that were never observed can be computed.

  place neurons   a population tiling the latent space; an item's location is a bump of activity over them, and
                  content is bound to location (Huber: a place cell is a what+where conjunction), so memories can be
                  retrieved *by position* ("what is two steps above X?")
  relations       each relation is a displacement ("taller than" = one step along some direction), learned from use
  one-shot        a new item is placed the moment its first relation arrives: position = known item + displacement
  re-anchoring    a fact that links two separate islands of knowledge shifts the whole smaller island so the fact
                  holds, then merges them (remapping: known memories adapted into one frame)
  consolidation   every observed relation is kept as an episode; replaying episodes applies a local delta rule that
                  moves both items (and the relation vector) toward consistency - scattered facts settle into one map,
                  and a mislabelled fact is outvoted by its neighbours instead of breaking a chain

No gradients: the only update is error = (pos[b] - pos[a]) - d_rel, applied to the three quantities involved.
"""
import math
import random

import torch


class PlaceCells:
    """Place neurons with Gaussian fields on a regular lattice over [-span, span]^dim."""

    def __init__(self, dim=2, per_dim=48, span=24.0, width=0.6, max_cells=20000, seed=0):
        if per_dim ** dim <= max_cells:  # a full lattice in low dimensions
            axes = torch.linspace(-span, span, per_dim)
            grids = torch.meshgrid(*[axes] * dim, indexing="ij")
            self.centers = torch.stack([g.reshape(-1) for g in grids], 1)
        else:  # a lattice would explode in many dimensions: scatter the same number of place fields at random
            g = torch.Generator().manual_seed(seed)
            self.centers = (torch.rand(max_cells, dim, generator=g) * 2 - 1) * span
        self.width = width * (2 * span / (per_dim - 1))

    def encode(self, pos):
        """Activity of every place neuron for positions (..., dim)."""
        d2 = ((pos[..., None, :] - self.centers) ** 2).sum(-1)
        return torch.exp(-d2 / (2 * self.width ** 2))


class CognitiveMap:
    def __init__(self, dim=2, lr=0.5, rel_lr=0.1, replay=30, robust=None, place=None, seed=0,
                 transform=False, mat_lr=0.05, anchor_targets=False):
        self.dim, self.lr, self.rel_lr, self.replay = dim, lr, rel_lr, replay
        # robust: the push from any one fact saturates at this size, so a single contradicted fact is outvoted as an
        # outlier instead of dragging its whole neighbourhood (a quadratic rule spreads every error)
        self.robust = robust
        self.rng = random.Random(seed)
        self.gen = torch.Generator().manual_seed(seed)
        self.names, self.index = [], {}
        self.pos = torch.zeros(0, dim)
        self.placed = torch.zeros(0, dtype=torch.bool)
        self.island = []  # which connected island of knowledge each item belongs to
        self.members = {}  # island -> its items, kept as facts arrive so merging need not scan
        self.rel = {}
        # A relation as a *transformation* rather than a step. A displacement says everything standing in
        # this relation is the same move away, which is right for "taller" and impossible for "is in
        # the country" - one vector cannot send every city to its own country. A matrix can: it may
        # project, rotate or collapse, so many things can be sent to one without the map tearing.
        self.transform, self.mat_lr = transform, mat_lr
        # Whether a relation may move the things it points at. Sending many things to one needs a
        # transformation that loses information, and the same loss lets it satisfy every fact by
        # squashing its targets together - perfectly learned and perfectly useless. Anchoring the
        # targets says: the country is where its own facts put it; the relation must reach it.
        self.anchor_targets = anchor_targets
        self.mat = {}
        self._bulk = False
        self._centre_cache = None
        self._centre_at = 0
        self.centroid_every = 64
        self.episodes = []
        self.weights = []
        self.repel = None
        self.place = place or PlaceCells(dim=dim)
        self._islands = 0

    # ---------------------------------------------------------------- learning
    def _item(self, name):
        if name not in self.index:
            self.index[name] = len(self.names)
            self.names.append(name)
            self.pos = torch.cat([self.pos, torch.zeros(1, self.dim)])
            self.placed = torch.cat([self.placed, torch.tensor([False])])
            self.island.append(-1)
        return self.index[name]

    def _relation(self, rel):
        if rel not in self.rel:
            # a new relation gets its own direction, as orthogonal as possible to the ones already known
            d = torch.randn(self.dim, generator=self.gen)
            for other in self.rel.values():
                d = d - (d @ other) / (other @ other) * other
            if d.norm() < 1e-3:
                d = torch.randn(self.dim, generator=self.gen)
            self.rel[rel] = d / d.norm()
        return self.rel[rel]

    def _fresh_spot(self):
        """An unconnected fact starts its own island, a few steps from the known map, until something links it."""
        self._islands += 1
        direction = torch.randn(self.dim, generator=self.gen)
        return self._centre() + 4.0 * direction / direction.norm()

    def _centre(self):
        """The centroid, but reused for a while when placing a corpus.

        A fresh island only needs somewhere roughly beside what is already known, so the exact centre does not
        matter - but computing it averages every placed item, and over the tens of thousands of islands a corpus
        produces that alone dominates the build. Incremental placement keeps the exact value; bulk placement
        refreshes it every `centroid_every` islands."""
        if not self._bulk:
            return self._centroid()
        if self._centre_cache is None or self._islands - self._centre_at >= self.centroid_every:
            self._centre_cache = self._centroid()
            self._centre_at = self._islands
        return self._centre_cache

    def _centroid(self):
        return self.pos[self.placed].mean(0) if bool(self.placed.any()) else torch.zeros(self.dim)

    def recenter(self):
        """Shift the whole map so its centre sits in the middle of the place-cell lattice. A global shift changes
        no relation; it keeps the map inside the region the place neurons cover (place cells remap to the space in
        use)."""
        if bool(self.placed.any()):
            self.pos[self.placed] -= self._centroid()

    def remap(self, scale=4.0, eps=1e-6):
        """Spread the map back out over the space the place neurons cover, without disturbing any fact.

        Consolidation alone has a trivial solution: put everything in one spot and every relation at zero length,
        which satisfies "a is like b" perfectly and says nothing. Relaxation drifts toward it whenever undirected
        facts outnumber directed ones. The escape is that positions only ever mean anything *relative* to each
        other: applying one linear map T to every position and every relation vector leaves every displacement
        exactly as satisfied as it was, because (x@T - y@T) = (x-y)@T. So the map is free to be re-expanded to
        unit spread along every axis - a remapping to the enclosure, not a correction - and the collapsing
        direction is removed on each pass. What survives is the structure that the facts actually pin down."""
        live = self.placed.nonzero().squeeze(1)
        if len(live) < 2:
            return
        self.pos[live] -= self.pos[live].mean(0)
        X = self.pos[live]
        C = (X.T @ X) / len(live) + eps * torch.eye(self.dim, device=X.device, dtype=X.dtype)
        evals, evecs = torch.linalg.eigh(C)
        T = evecs @ torch.diag(scale * evals.clamp_min(eps).rsqrt()) @ evecs.T  # C^(-1/2), symmetric
        self.pos = self.pos @ T
        for k in self.rel:
            self.rel[k] = self.rel[k] @ T

    def downscale(self, factor=0.98, floor=0.05):
        """Sleep's other job: scale everything down a little, and drop what falls through.

        The synaptic homeostasis account of sleep is that waking drives synapses up across the board and sleep
        scales them back proportionally, so relative strengths survive while the accumulated floor of weak,
        accidental associations does not - the signal is kept and the noise is renormalised away. Here the
        analogue is uniform: multiply every position and every relation vector by the same factor, which changes
        no relation at all (a scale is the same linear map applied to both sides), then forget the episodes whose
        residual is so large that they never fitted anything. Unlike inhibition this pushes nothing apart; it
        removes what was never holding together."""
        self.pos *= factor
        for k in self.rel:
            self.rel[k] = self.rel[k] * factor
        if floor and self.episodes:
            keep, kept_w = [], []
            for (a, moves, b), w in zip(self.episodes, self.weights or [1.0] * len(self.episodes)):
                d = self._disp(moves) if moves else torch.zeros(self.dim, device=self.pos.device)
                if float(((self.pos[b] - self.pos[a]) - d).norm()) <= floor or w >= 1.0:
                    keep.append((a, moves, b))
                    kept_w.append(w)
            dropped = len(self.episodes) - len(keep)
            if dropped and len(keep) > 10:
                self.episodes, self.weights = keep, kept_w
            return dropped
        return 0

    def separate_relations(self, strength=0.5, floor=0.15):
        """Keep different relations pointing in different directions.

        They are *made* orthogonal, and then consolidation is free to drag them together: whatever direction
        best fits one relation's facts also partly fits another's, and nothing pushes back. Measured on the
        built map, HasPrerequisite, MotivatedByGoal and HasFirstSubevent had collapsed onto one another at a
        cosine of 1.00, and DerivedFrom onto IsA at 0.99.

        That is precisely why the relation labels came out wrong. If two relations are the same direction then
        stepping along either lands in the same place, and which name gets attached to the answer is a coin
        toss - "a doctor is part of a medical practitioner" is that coin landing badly.

        Distinct relations need distinct codes, and nothing in the map was keeping them distinct. This is
        lateral inhibition again, applied to the directions rather than to the things: relations that have
        grown too alike push each other apart.
        """
        names = sorted(self.rel)
        if len(names) < 2:
            return 0.0
        M = torch.stack([self.rel[k] for k in names])
        norms = M.norm(dim=1, keepdim=True).clamp_min(1e-9)
        U = M / norms
        cos = U @ U.T
        cos.fill_diagonal_(0.0)
        # push each relation away from the others in proportion to how far it has drifted onto them
        push = (cos.abs() * torch.sign(cos)) @ U
        U = U - strength * push / max(1, len(names) - 1)
        U = U / U.norm(dim=1, keepdim=True).clamp_min(1e-9)
        for i, k in enumerate(names):
            self.rel[k] = U[i] * norms[i].clamp_min(floor)
        after = (U @ U.T)
        after.fill_diagonal_(0.0)
        return float(after.abs().mean())

    def inhibit(self, pairs=4096, margin=1.0, strength=0.2, repel=None):
        """Lateral inhibition: sample random pairs of items and push apart any that have crowded closer than
        `margin`. This is what keeps the map from collapsing, and it is the same kind of rule as everything else
        here - local, pairwise, error-driven, no gradient and no global decomposition.

        It is needed because "a is like b" is satisfied perfectly by putting a and b in the same spot, so a map
        told mostly about likeness drifts toward a single point that says nothing. Two items being *distinct* is
        itself information the facts never state, and inhibition is how the population asserts it: neighbouring
        fields that fire together for everything separate until they don't. Only crowded pairs are touched, so
        items the facts genuinely place together stay together - the push is a floor, not a spring."""
        live = self.placed.to(self.pos.device).nonzero().squeeze(1)
        if len(live) < 2:
            return 0.0
        repel = None if repel is None else repel.to(self.pos.device)
        # the map's own generator lives on the CPU; off it, fall back to the global stream
        g = self.gen if self.pos.device.type == "cpu" else None
        pick = lambda: live[torch.randint(len(live), (pairs,), generator=g, device=self.pos.device)]
        i, j = pick(), pick()
        if repel is not None and len(repel):  # pairs a fact says are unlike: always pushed, never sampled away
            i = torch.cat([i, repel[:, 0]])
            j = torch.cat([j, repel[:, 1]])
        keep = i != j
        i, j = i[keep], j[keep]
        diff = self.pos[i] - self.pos[j]
        dist = diff.norm(dim=1)
        crowded = dist < margin
        if not bool(crowded.any()):
            return 0.0
        i, j, diff, dist = i[crowded], j[crowded], diff[crowded], dist[crowded]
        # a random direction for pairs sitting exactly on top of each other, which have no direction to push along
        rand = torch.randn(diff.shape, generator=g, device=self.pos.device)
        unit = torch.where(dist[:, None] > 1e-6, diff / dist[:, None].clamp_min(1e-6),
                           rand / rand.norm(dim=1, keepdim=True).clamp_min(1e-6))
        push = strength * (margin - dist)[:, None] * unit
        self.pos.index_add_(0, i, 0.5 * push)
        self.pos.index_add_(0, j, -0.5 * push)
        return float(crowded.float().mean())

    def _matrix(self, rel):
        """The transformation for a relation, starting as identity so it begins as a pure displacement and only
        departs from that as far as the facts push it."""
        if rel not in self.mat:
            self.mat[rel] = torch.eye(self.dim)
        return self.mat[rel]

    def predict(self, a_pos, rel):
        """Where a relation says you land, starting from here."""
        if self.transform:
            return a_pos @ self._matrix(rel).T + self._relation(rel)
        return a_pos + self._relation(rel)

    def _learn_transform(self, ia, rel, ib, lr):
        """One fact, learned locally.

        error = where b is, minus where the relation said b would be. The matrix is corrected by that error
        times the input it was applied to - the delta rule for a linear associator, which is Hebbian with a sign
        and needs no backward pass. The two items move as well, b toward the prediction and a along the
        transposed weights, which is what a reciprocal connection does in a network.
        """
        M = self._matrix(rel)
        d = self._relation(rel)
        a, b = self.pos[ia], self.pos[ib]
        err = b - (a @ M.T + d)
        if self.robust:
            err = err * min(1.0, self.robust / (float(err.norm()) + 1e-9))
        scale = float(a @ a) + 1e-6
        self.mat[rel] = M + self.mat_lr * torch.outer(err, a) / scale
        step = d + self.rel_lr * err
        # the same floor the displacement path has always had. Without it the offset shrinks toward nothing,
        # which fits every fact and means nothing - the collapse this map has had to be defended from at every
        # level. Leaving it out is why the transformation lost even with its matrix held at the identity.
        self.rel[rel] = step if float(step.norm()) > 0.2 else d
        if not self.anchor_targets:
            self.pos[ib] = b - 0.5 * lr * err
        # the division by the input's size belongs to the matrix update, which has to be scale-free;
        # applying it here as well shrank every position step by the square of the map's spread, and
        # with the matrix frozen at the identity that alone made this path lose to the one it copies
        self.pos[ia] = a + (1.0 if self.anchor_targets else 0.5) * lr * (err @ M)

    def _disp(self, moves):
        """Displacement of a combination of moves, e.g. (("east", 1), ("north", 2))."""
        return sum((count * self.rel[r] for r, count in moves), torch.zeros(self.dim))

    def _consolidate(self, a, moves, b, lr):
        moves = ((moves, 1),) if isinstance(moves, str) else moves
        if self.transform and len(moves) == 1 and moves[0][1] == 1:
            return self._learn_transform(a, moves[0][0], b, lr)
        d = self._disp(moves)
        err = (self.pos[b] - self.pos[a]) - d
        if self.robust:
            err = err * min(1.0, self.robust / (float(err.norm()) + 1e-9))
        self.pos[a] += 0.5 * lr * err
        self.pos[b] -= 0.5 * lr * err
        if self.rel_lr and len(moves) == 1 and moves[0][1] == 1:  # single-step facts teach the relation itself
            rel = moves[0][0]
            step = self.rel_lr * err
            self.rel[rel] = d + step if (d + step).norm() > 0.2 else d

    def observe(self, a, rel, b, count=1):
        """One fact: `b` lies `count` `rel`-steps from `a` (a='Bob', rel='height', b='Alice': Alice is taller)."""
        self.observe_moves(a, ((rel, count),), b)

    def distinguish(self, pairs):
        """Facts that say two things are *unlike* (antonyms, "is distinct from"). Co-locating them would be the
        opposite of what the fact says, and a displacement pointing one way would be just as wrong, so they are
        recorded as pairs that inhibition always pushes apart."""
        idx = [(self._item(a), self._item(b)) for a, b in pairs]
        self.repel = torch.tensor(idx, dtype=torch.long) if idx else None
        return self

    def observe_moves(self, a, moves, b, weight=1.0):
        """One fact made of several moves: b = a + sum(count * relation), e.g. 2 north and 1 east."""
        moves = tuple((r, c) for r, c in moves if c)
        for r, _ in moves:
            self._relation(r)
        ia, ib, d = self._item(a), self._item(b), self._disp(moves)
        if not self.placed[ia] and not self.placed[ib]:
            self.pos[ia] = self._fresh_spot()
            self.pos[ib] = self.pos[ia] + d
            self.island[ia] = self.island[ib] = self._islands
            self.members[self._islands] = [ia] if ia == ib else [ia, ib]
        elif not self.placed[ib]:
            self.pos[ib] = self.pos[ia] + d
            self.island[ib] = self.island[ia]
            self.members.setdefault(self.island[ia], []).append(ib)
        elif not self.placed[ia]:
            self.pos[ia] = self.pos[ib] - d
            self.island[ia] = self.island[ib]
            self.members.setdefault(self.island[ib], []).append(ia)
        elif self.island[ia] != self.island[ib]:
            self._reanchor(ia, ib, d)
        else:
            self._consolidate(ia, moves, ib, self.lr)
        self.placed[ia] = self.placed[ib] = True
        self.episodes.append((ia, moves, ib))
        self.weights.append(float(weight))
        for _ in range(self.replay):  # replay: consolidation over remembered episodes
            ea, er, eb = self.episodes[self.rng.randrange(len(self.episodes))]
            self._consolidate(ea, er, eb, self.lr)

    def attract(self, a, b, weight=1.0):
        """A symmetric fact ("x is like y"): no direction, just co-location. A displacement that means "is like"
        would be wrong pointing either way, so the fact is recorded as a zero displacement and the same delta rule
        pulls the two together. Directed relations keep their norm-floored directions, so the map still spreads
        out instead of collapsing to a point."""
        self.observe_moves(a, (), b, weight=weight)

    def observe_bulk(self, edges, progress=None):
        """Place many facts at once. The per-fact replay that `observe_moves` does is what makes one-shot learning
        settle immediately; over a corpus it is both unaffordable and unnecessary, because `sleep()` replays every
        episode in one batched pass afterwards. Each edge is (a, moves, b); moves=() means attract."""
        edges = list(edges)
        fresh = []
        for edge in edges:      # every name the corpus will mention, allocated once
            for name in (edge[0], edge[2]):
                if name not in self.index:
                    self.index[name] = len(self.names) + len(fresh)
                    fresh.append(name)
        if fresh:
            self.names.extend(fresh)
            self.pos = torch.cat([self.pos, torch.zeros(len(fresh), self.dim)])
            self.placed = torch.cat([self.placed, torch.zeros(len(fresh), dtype=torch.bool)])
            self.island.extend([-1] * len(fresh))
        replay, self.replay = self.replay, 0
        self._bulk, self._centre_cache = True, None
        try:
            for i, edge in enumerate(edges):
                a, moves, b = edge[0], edge[1], edge[2]
                self.observe_moves(a, moves, b, weight=edge[3] if len(edge) > 3 else 1.0)
                if progress and i and i % progress == 0:
                    print(f"  placed {i:,} facts, {len(self.names):,} items, {self._islands:,} islands", flush=True)
        finally:
            self.replay = replay
            self._bulk = False
        return self

    def _reanchor(self, ia, ib, d):
        """Two islands just got linked: move the smaller one rigidly so the new fact holds, and merge them."""
        ka, kb = self.island[ia], self.island[ib]   # read the ids before anything overwrites them
        A = self.members.setdefault(ka, [ia])
        B = self.members.setdefault(kb, [ib])
        shift = (self.pos[ia] + d) - self.pos[ib]
        if len(B) <= len(A):
            self.pos[B] += shift
            keep, gone, move = ka, kb, B
        else:
            self.pos[A] -= shift
            keep, gone, move = kb, ka, A
        for i in move:
            self.island[i] = keep
        self.members[keep] = self.members.get(keep, []) + move
        self.members.pop(gone, None)

    def consolidate_transform(self, rounds=80, lr=0.15, target_rate=0.15, tolerance=0.5,
                              device=None, verbose=False, inhibit=True, margin=1.0,
                              strength=0.2, pairs=8192, shrink=0.0):
        """Consolidation for relations held as transformations, over a whole corpus at once.

        The same rule as `_learn_transform`, applied to every fact of a relation together instead of one at a
        time: the matrix is corrected by the errors times the inputs they were applied to, which batched is one
        matrix product. `target_rate` is how far a relation may move the things it points at - zero pins them
        entirely, which is what stops a many-to-one relation from satisfying itself by squashing its targets,
        and a little above zero lets a thing that is only ever a target still settle.
        """
        if not self.episodes:
            return
        home = self.pos.device
        if device is not None:
            self.pos = self.pos.to(device)
            self.rel = {k: v.to(device) for k, v in self.rel.items()}
            self.mat = {k: v.to(device) for k, v in self.mat.items()}
            self.placed = self.placed.to(device)
            if self.repel is not None:
                self.repel = self.repel.to(device)

        by_rel = {}
        for ia, moves, ib in self.episodes:
            if len(moves) != 1 or moves[0][1] != 1:
                continue
            by_rel.setdefault(moves[0][0], ([], []))
            by_rel[moves[0][0]][0].append(ia)
            by_rel[moves[0][0]][1].append(ib)
        grouped = {r: (torch.tensor(a, device=self.pos.device), torch.tensor(b, device=self.pos.device))
                   for r, (a, b) in by_rel.items() if len(a) > 1}
        if verbose:
            print(f"  {len(grouped)} relations, {sum(len(a) for a, _ in grouped.values()):,} single-step facts",
                  flush=True)

        # Inhibition's margin has to mean something relative to how spread out the map actually is. Given a
        # fixed margin of 1 on a map whose items sit 0.2 apart, every pair counts as crowded and the whole map
        # is pushed outward every round - which is exactly what happened: it expanded to a spread of ninety.
        live = self.placed.nonzero().squeeze(1)[:600]
        if len(live) > 2 and margin:
            typical = float(torch.cdist(self.pos[live], self.pos[live]).median())
            margin = margin * max(typical, 1e-6)
            if verbose:
                print(f"  inhibition margin scaled to {margin:.4f} for a map spaced {typical:.4f}", flush=True)

        for step in range(rounds):
            total = count = 0.0
            # every item is pulled once per fact it takes part in, so a thing in a hundred thousand facts
            # would be kicked a hundred thousand times a round. What each item wants is the *average* of what
            # its facts ask of it, which is what the displacement path has always computed.
            delta = torch.zeros_like(self.pos)
            hits = torch.zeros(len(self.pos), 1, device=self.pos.device)
            for rel, (ia, ib) in grouped.items():
                M, d = self._matrix(rel).to(self.pos.device), self._relation(rel).to(self.pos.device)
                A, B = self.pos[ia], self.pos[ib]
                err = B - (A @ M.T + d)
                w = torch.clamp(tolerance / (err.norm(dim=1) + 1e-9), max=1.0)[:, None]
                err = err * w
                total += float(err.norm(dim=1).sum())
                count += len(ia)
                scale = float((A * A).sum()) + 1e-6
                new_M = M + self.mat_lr * (err.T @ A) / scale
                if shrink:
                    # A full matrix is four thousand numbers per relation against a few thousand facts, and
                    # on held-out facts every setting of it did worse than a plain step. Pulling it back
                    # toward the identity each round says: be a displacement unless the facts insist
                    # otherwise, and pay for every departure.
                    new_M = (1.0 - shrink) * new_M + shrink * torch.eye(
                        self.dim, device=self.pos.device, dtype=new_M.dtype)
                self.mat[rel] = new_M
                nd = d + self.rel_lr * err.mean(0)
                self.rel[rel] = nd if float(nd.norm()) > 0.2 else d
                ones = torch.ones(len(ia), 1, device=self.pos.device)
                delta.index_add_(0, ia, err @ M)
                hits.index_add_(0, ia, ones)
                if target_rate:
                    delta.index_add_(0, ib, -target_rate * err)
                    hits.index_add_(0, ib, ones * target_rate)
            self.pos += lr * delta / hits.clamp_min(1.0)
            if inhibit:
                self.inhibit(pairs=pairs, margin=margin, strength=strength, repel=self.repel)
            if verbose and step % max(1, rounds // 10) == 0:
                print(f"  round {step:4d}/{rounds}  residual {total / max(1, count):.4f}  "
                      f"spread {float(self.pos[self.placed].std()):.3f}", flush=True)

        if device is not None:
            self.pos = self.pos.to(home)
            self.rel = {k: v.to(home) for k, v in self.rel.items()}
            self.mat = {k: v.to(home) for k, v in self.mat.items()}
            self.placed = self.placed.to(home)
            if self.repel is not None:
                self.repel = self.repel.to(home)

    def sleep(self, rounds=200, damping=0.5, tolerance=0.5, device=None, verbose=False, spread=None,
              inhibit=None, margin=1.0, strength=0.2, pairs=4096, schema=0.0, shy=0.0,
              rel_floor=0.0, separate=0.0):
        """Offline consolidation: every remembered episode is replayed at once (like sleep ripples) and every item
        moves toward where its neighbours, taken together, say it should be. Local - each item hears only its own
        neighbours - but the network relaxes as a whole, so a region misplaced by one bad fact is pulled back as a
        unit. Facts that disagree with the consensus by more than `tolerance` are down-weighted (robust: a lone
        mislabelled fact is outvoted rather than averaged in). Relation vectors settle to the average displacement
        of the facts that carry them."""
        if not self.episodes:
            return
        self.recenter()
        home = self.pos.device
        if device is not None:  # the relaxation is pure batched tensor work; it runs anywhere
            self.pos = self.pos.to(device)
            self.rel = {k: v.to(device) for k, v in self.rel.items()}
            self.placed = self.placed.to(device)
            if self.repel is not None:
                self.repel = self.repel.to(device)
        a = torch.tensor([e[0] for e in self.episodes], device=self.pos.device)
        b = torch.tensor([e[2] for e in self.episodes], device=self.pos.device)
        rels = sorted(self.rel)
        C = torch.zeros(len(self.episodes), len(rels), device=self.pos.device)  # how many steps of each relation every episode spans
        for i, (_, moves, _) in enumerate(self.episodes):
            for rname, count in moves:
                C[i, rels.index(rname)] += count
        fact_w = torch.tensor(self.weights or [1.0] * len(self.episodes), device=self.pos.device)[:, None]
        unit = (C.abs().sum(1) == 1) & (C.max(1).values == 1)  # plain single-step facts
        r = C.argmax(1)
        for _ in range(rounds):
            D = C @ torch.stack([self.rel[k] for k in rels])
            res = (self.pos[b] - self.pos[a]) - D
            w = torch.clamp(tolerance / (res.norm(dim=1) + 1e-9), max=1.0)[:, None] * fact_w  # Huber x fact
            if schema:  # what already fits the structure is taken up faster (schema-consistent memory)
                w = w * torch.exp(-schema * res.norm(dim=1))[:, None]
            num = torch.zeros_like(self.pos)
            den = torch.zeros(len(self.pos), 1, device=self.pos.device)
            num.index_add_(0, b, w * (self.pos[a] + D))
            num.index_add_(0, a, w * (self.pos[b] - D))
            den.index_add_(0, b, w)
            den.index_add_(0, a, w)
            has = den[:, 0] > 0
            self.pos[has] = (1 - damping) * self.pos[has] + damping * num[has] / den[has]
            if self.rel_lr:
                disp = self.pos[b] - self.pos[a]
                for k, name in enumerate(rels):
                    sel = unit & (r == k)
                    if not bool(sel.any()):
                        continue
                    avg = (w[sel] * disp[sel]).sum(0) / w[sel].sum().clamp_min(1e-9)
                    self.rel[name] = (1 - self.rel_lr) * self.rel[name] + self.rel_lr * avg
                    if rel_floor:
                        # a relation that means something has to be a real step. Facts that disagree about a
                        # relation are best fitted by shrinking it to nothing - which fits well and says nothing,
                        # the same trade the whole map makes about likeness - so hold it to a minimum length.
                        n = self.rel[name].norm()
                        if 0 < float(n) < rel_floor:
                            self.rel[name] = self.rel[name] * (rel_floor / n)
            if inhibit:
                self.inhibit(pairs=pairs, margin=margin, strength=strength, repel=self.repel)
            if separate:
                self.separate_relations(strength=separate)
            if shy:
                self.downscale(factor=1.0 - shy, floor=0.0)
            elif spread:
                self.remap(scale=spread)
            if verbose and _ % max(1, rounds // 10) == 0:
                print(f"  sleep {_:4d}/{rounds}  residual {float(res.norm(dim=1).mean()):.4f}", flush=True)
        if device is not None:
            self.pos = self.pos.to(home)
            self.rel = {k: v.to(home) for k, v in self.rel.items()}
            self.placed = self.placed.to(home)
            if self.repel is not None:
                self.repel = self.repel.to(home)

    # ---------------------------------------------------------------- reasoning
    def knows(self, name):
        return name in self.index and bool(self.placed[self.index[name]])

    def steps(self, a, rel, b):
        """How many `rel`-steps b lies from a (positive: b is `rel` of a). The magnitude doubles as confidence -
        which is why far-apart items are compared more reliably (the symbolic distance effect)."""
        d = self.rel[rel]
        return float((self.pos[self.index[b]] - self.pos[self.index[a]]) @ d / (d @ d))

    def at(self, a, moves, top=3):
        """Items nearest to the position reached from `a` by the given moves [(rel, count), ...] - retrieval by
        location through the place neurons, not by any stored link."""
        target = self.pos[self.index[a]].clone()
        for rel, count in moves:
            target += count * self.rel[rel]
        return self.nearest(target, top)

    def nearest(self, target, top=3):
        """Place-cell readout: the items whose place codes overlap most with the code of the target position. For
        Gaussian place fields tiling the space, that overlap falls monotonically with distance (the product of two
        Gaussian bumps integrates to a Gaussian of their separation), so ranking by distance is the exact readout -
        and it keeps working at the lattice's edge, where individual place cells run out."""
        live = self.placed.nonzero().squeeze(1)
        if not len(live):
            return []
        dist = (self.pos[live] - target).norm(dim=1)
        order = dist.argsort()[:top]
        return [(self.names[int(live[i])], float(dist[i])) for i in order]

    def between(self, a, b, rel):
        """Items whose position along `rel` lies strictly between a and b."""
        lo, hi = sorted((0.0, self.steps(a, rel, b)))
        return [n for n in self.names if n not in (a, b) and self.knows(n) and lo + 0.5 < self.steps(a, rel, n) < hi - 0.5]

    def ranking(self, rel):
        """Every placed item ordered along `rel`."""
        d = self.rel[rel]
        live = [n for n in self.names if self.knows(n)]
        return sorted(live, key=lambda n: float(self.pos[self.index[n]] @ d))

    def footprint(self):
        return int(self.placed.sum()) * self.dim + len(self.rel) * self.dim

In [ ]:
%%writefile /kaggle/working/neuromoe/map_bench.py
"""Pseudospatial memory vs memory-only chaining vs trained sequence models (PREREGISTRATION_map.md).

  transitive  N items taught only through adjacent pairs ("hi is above lo"), shuffled, each shown once, a fraction
              `noise` of them mislabelled (reversed). Test: every non-adjacent pair, by symbolic distance.
  grid2d      a W x W grid of items taught only through neighbour relations (east / north), shuffled, once, some
              mislabelled. Test: "from X, move dx east and dy north - which item is there?" for |dx|+|dy| >= 2, i.e.
              places never related to X directly.

  map        cogmap.CognitiveMap: relations as displacements, one-shot placement, re-anchoring, replay (no gradients)
  chain      NeuroMoE working memory only: one-shot bindings per relation, answered by following links (no map)
  gru / transformer   trained by backprop on thousands of such episodes, answering in context from the same facts

  python map_bench.py --seeds 10 --steps 6000 [--device cuda]
"""
import argparse
import json
import math
import os
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from cogmap import CognitiveMap
from neuromoe import SparseExpansion, WorkingMemory

HERE = os.path.dirname(os.path.abspath(__file__))
SYMBOLS = 200


# ------------------------------------------------------------------------------------------------------ tasks
def transitive_episode(N, noise, rng):
    """Returns (items low->high, facts [(lo, hi)] as shown - possibly reversed -, queries [(x, y, y_above_x, dist)])."""
    items = rng.sample(range(SYMBOLS), N)
    facts = []
    for k in range(N - 1):
        lo, hi = items[k], items[k + 1]
        facts.append((hi, lo) if rng.random() < noise else (lo, hi))
    rng.shuffle(facts)
    queries = [(items[i], items[j], j > i, abs(i - j)) for i in range(N) for j in range(N) if abs(i - j) >= 2]
    return items, facts, queries


def grid_episode(W, noise, rng):
    """Returns (grid[x][y] items, facts [(a, rel, b)] meaning b is one `rel` step from a, queries [(X, dx, dy, answer)])."""
    ids = rng.sample(range(SYMBOLS), W * W)
    grid = [[ids[x * W + y] for y in range(W)] for x in range(W)]
    facts = []
    for x in range(W):
        for y in range(W):
            for rel, (nx, ny) in (("east", (x + 1, y)), ("north", (x, y + 1))):
                if nx < W and ny < W:
                    a, b = grid[x][y], grid[nx][ny]
                    facts.append((b, rel, a) if rng.random() < noise else (a, rel, b))
    rng.shuffle(facts)
    queries = [(grid[x][y], tx - x, ty - y, grid[tx][ty]) for x in range(W) for y in range(W)
               for tx in range(W) for ty in range(W) if abs(tx - x) + abs(ty - y) >= 2]
    return grid, facts, queries


# ------------------------------------------------------------------------------------------------------ learners
def new_map(cfg, seed):
    return CognitiveMap(dim=2, lr=cfg["lr"], rel_lr=cfg["rel_lr"], replay=cfg["replay"], robust=cfg["robust"],
                        seed=seed)


def map_transitive(facts, queries, cfg, seed):
    m = new_map(cfg, seed)
    for lo, hi in facts:
        m.observe(str(lo), "above", str(hi))
    m.sleep(cfg["sleep"], tolerance=cfg["tol"])
    return [(m.steps(str(x), "above", str(y)) > 0) == above for x, y, above, _ in queries]


def map_grid(facts, queries, cfg, seed):
    m = new_map(cfg, seed)
    for a, rel, b in facts:
        m.observe(str(a), rel, str(b))
    m.sleep(cfg["sleep"], tolerance=cfg["tol"])
    out = []
    for X, dx, dy, ans in queries:
        best = m.at(str(X), [("east", dx), ("north", dy)], top=1)
        out.append(bool(best) and best[0][0] == str(ans))
    return out


class ChainMemory:
    """Memory-only NeuroMoE: each relation is a one-shot associative store (sparse key -> next item), in both
    directions. Questions are answered by following stored links, one recall at a time."""

    def __init__(self, relations, n=2048, k=32, seed=0):
        self.codes = SparseExpansion(SYMBOLS, n=n, k=k, seed=seed)(torch.eye(SYMBOLS))
        self.eye = torch.eye(SYMBOLS + 1)  # the extra slot means "nothing stored here"
        self.mem = {(r, s): WorkingMemory(n, SYMBOLS + 1) for r in relations for s in (1, -1)}
        self.known = {key: set() for key in self.mem}

    def observe(self, a, rel, b):
        self.mem[(rel, 1)].write(self.codes[a], self.eye[b])
        self.mem[(rel, -1)].write(self.codes[b], self.eye[a])
        self.known[(rel, 1)].add(a)
        self.known[(rel, -1)].add(b)

    def step(self, x, rel, sign):
        if x not in self.known[(rel, sign)]:
            return None
        return int(self.mem[(rel, sign)].read(self.codes[x]).argmax())

    def walk(self, x, rel, sign, n):
        for _ in range(n):
            x = self.step(x, rel, sign)
            if x is None:
                return None
        return x


def chain_transitive(facts, queries, N, rng):
    c = ChainMemory(["above"])
    for lo, hi in facts:
        c.observe(lo, "above", hi)
    out = []
    for x, y, above, _ in queries:
        verdict = None
        for sign in (1, -1):  # search upward, then downward, following links
            z = x
            for _ in range(N):
                z = c.step(z, "above", sign)
                if z is None:
                    break
                if z == y:
                    verdict = sign == 1
                    break
            if verdict is not None:
                break
        out.append((verdict if verdict is not None else rng.random() < 0.5) == above)
    return out


def chain_grid(facts, queries):
    c = ChainMemory(["east", "north"])
    for a, rel, b in facts:
        c.observe(a, rel, b)
    out = []
    for X, dx, dy, ans in queries:
        z = c.walk(X, "east", 1 if dx > 0 else -1, abs(dx))
        z = c.walk(z, "north", 1 if dy > 0 else -1, abs(dy)) if z is not None else None
        out.append(z == ans)
    return out


# ------------------------------------------------------------------------------------------------------ trained baselines
QUERY, ABOVE_TOK = SYMBOLS, SYMBOLS + 1
EAST, NORTH, ASK = SYMBOLS, SYMBOLS + 1, SYMBOLS + 2
DX0, DY0 = SYMBOLS + 3 + 4, SYMBOLS + 12 + 4  # offset tokens for -4..4
VOCAB = SYMBOLS + 21


class TinyTransformer(nn.Module):
    def __init__(self, n_out, d=128, layers=4, heads=4, max_len=768, dropout=0.0):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, d)
        self.pos = nn.Parameter(0.02 * torch.randn(max_len, d))
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, activation="gelu", batch_first=True,
                                           norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.out = nn.Linear(d, n_out)

    def forward(self, tok):
        L = tok.shape[1]
        mask = torch.triu(torch.full((L, L), float("-inf"), device=tok.device), 1)
        return self.out(self.blocks(self.emb(tok) + self.pos[:L], mask=mask, is_causal=True))


class GRUNet(nn.Module):
    def __init__(self, n_out, hidden=256, emb=64):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, emb)
        self.gru = nn.GRU(emb, hidden, batch_first=True)
        self.out = nn.Linear(hidden, n_out)

    def forward(self, tok):
        return self.out(self.gru(self.emb(tok))[0])


def transitive_tokens(facts, queries, max_q=None, rng=None):
    tok = [t for lo, hi in facts for t in (lo, hi)]
    tgt = [-1] * len(tok)
    qs = queries if max_q is None else rng.sample(queries, min(max_q, len(queries)))
    for x, y, above, _ in qs:
        tok += [QUERY, x, y]
        tgt += [-1, -1, int(above)]
    return tok, tgt, qs


def grid_tokens(facts, queries, max_q=None, rng=None):
    tok = [t for a, rel, b in facts for t in (a, EAST if rel == "east" else NORTH, b)]
    tgt = [-1] * len(tok)
    qs = queries if max_q is None else rng.sample(queries, min(max_q, len(queries)))
    for X, dx, dy, ans in qs:
        tok += [ASK, X, DX0 + dx, DY0 + dy]
        tgt += [-1, -1, -1, ans]
    return tok, tgt, qs


def batchify(seqs, device):
    L = max(len(t) for t, _ in seqs)
    tok = torch.full((len(seqs), L), 0, dtype=torch.long)
    tgt = torch.full((len(seqs), L), -1, dtype=torch.long)
    for i, (t, g) in enumerate(seqs):
        tok[i, :len(t)] = torch.tensor(t)
        tgt[i, :len(g)] = torch.tensor(g)
    return tok.to(device), tgt.to(device)


def train_net(net, make_seq, steps, lr, device, batch=32):
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=max(steps, 1), pct_start=0.05)
    net.train()
    for _ in range(steps):
        tok, tgt = batchify([make_seq() for _ in range(batch)], device)
        loss = F.cross_entropy(net(tok).reshape(-1, net.out.out_features), tgt.reshape(-1), ignore_index=-1)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
        sched.step()
    net.eval()
    return net


@torch.no_grad()
def net_answers(net, to_tokens, facts, queries, device, chunk=24):
    """The same facts, then queries in chunks of the size seen in training; answers in query order."""
    hits = []
    for i in range(0, len(queries), chunk):
        tok, tgt, _ = to_tokens(facts, queries[i:i + chunk])
        t, g = batchify([(tok, tgt)], device)
        pred = net(t).argmax(-1)[0]
        mask = g[0] >= 0
        hits += (pred[mask] == g[0][mask]).cpu().tolist()
    return hits


# ------------------------------------------------------------------------------------------------------ bench
MAP_GRID = [dict(lr=lr, rel_lr=rl, replay=rp, robust=rb, sleep=sl, tol=tol) for lr in (0.3, 0.6)
            for rl in (0.0, 0.05) for rp in (10, 40) for rb in (None, 0.3) for sl in (100, 300) for tol in (0.3, 0.6)]
NOISES, T_SIZES, W_SIZES = (0.0, 0.05, 0.1), (8, 16, 32), (4, 5)


def by_distance(hits, queries):
    d = {}
    for h, q in zip(hits, queries):
        d.setdefault(q[3], []).append(h)
    return {k: float(np.mean(v)) for k, v in sorted(d.items())}


def slope(acc_by_dist):
    xs = np.array(list(acc_by_dist.keys()), dtype=float)
    ys = np.array(list(acc_by_dist.values()))
    return float(np.polyfit(xs, ys, 1)[0]) if len(xs) > 1 else 0.0


def tune_map(val_seeds=range(1000, 1008)):
    """Map settings chosen on validation episodes only (different seeds from every reported episode), on both tasks."""
    def score(cfg):
        accs = []
        for s in val_seeds:
            rng = random.Random(s)
            _, facts, qs = transitive_episode(16, 0.1, rng)
            accs.append(np.mean(map_transitive(facts, qs, cfg, s)))
            _, gfacts, gqs = grid_episode(5, 0.1, rng)
            accs.append(np.mean(map_grid(gfacts, gqs, cfg, s)))
        return float(np.mean(accs))
    scored = [(score(c), c) for c in MAP_GRID]
    best = max(scored, key=lambda sc: sc[0])
    print(f"map tuned on validation: {best[1]} (val acc {best[0]:.3f})", flush=True)
    return best[1]


def train_baselines(steps, device, lr=1e-3):
    """Transformer and GRU trained on fresh episodes of every size and noise level used in testing."""
    rng = random.Random(12345)
    nets = {}
    for name, make in (("transformer", lambda n: TinyTransformer(n)), ("gru", lambda n: GRUNet(n))):
        t0 = time.time()
        tnet = train_net(make(2).to(device), lambda: transitive_tokens(*transitive_episode(
            rng.choice(range(6, 33)), rng.choice(NOISES), rng)[1:], max_q=24, rng=rng)[:2], steps, lr, device)
        gnet = train_net(make(SYMBOLS).to(device), lambda: grid_tokens(*grid_episode(
            rng.choice((3, 4, 5)), rng.choice(NOISES), rng)[1:], max_q=24, rng=rng)[:2], steps, lr, device)
        nets[name] = (tnet, gnet)
        print(f"trained {name} ({steps} steps each task) in {time.time() - t0:.0f}s", flush=True)
    return nets


def run(seeds, steps, device):
    cfg = tune_map()
    nets = train_baselines(steps, device) if steps else {}
    rows = []
    for s in range(seeds):
        row = {"seed": s}
        for noise in NOISES:
            for N in T_SIZES:
                rng = random.Random(10_000 * s + int(noise * 100) + N)
                _, facts, qs = transitive_episode(N, noise, rng)
                results = {"map": map_transitive(facts, qs, cfg, s), "chain": chain_transitive(facts, qs, N, rng)}
                for name, (tnet, _) in nets.items():
                    results[name] = net_answers(tnet, transitive_tokens, facts, qs, device)
                for name, hits in results.items():
                    tag = f"transitive_N{N}_noise{noise}_{name}"
                    dist = by_distance(hits, qs)
                    row[f"{tag}_acc"] = float(np.mean(hits))
                    row[f"{tag}_slope"] = slope(dist)
                    row[f"{tag}_bydist"] = dist
            for W in W_SIZES:
                rng = random.Random(20_000 * s + int(noise * 100) + W)
                _, facts, qs = grid_episode(W, noise, rng)
                results = {"map": map_grid(facts, qs, cfg, s), "chain": chain_grid(facts, qs)}
                for name, (_, gnet) in nets.items():
                    results[name] = net_answers(gnet, grid_tokens, facts, qs, device)
                for name, hits in results.items():
                    row[f"grid_W{W}_noise{noise}_{name}_acc"] = float(np.mean(hits))
        rows.append(row)
        print(f"seed {s} done", flush=True)
    return rows, cfg


PRIMARY = [  # id, metric key template, A, B  (paired over seeds, two-sided, Holm across all four)
    ("M1", "transitive_N16_noise0.1_{}_acc", "map", "chain"),
    ("M2", "transitive_N16_noise0.1_{}_slope", "map", "chain"),
    ("M3", "transitive_N16_noise0.1_{}_acc", "map", "transformer"),
    ("M4", "grid_W5_noise0.1_{}_acc", "map", "chain"),
]


def analyze(rows):
    from scipy import stats
    out, ps = ["# Pseudospatial memory: pre-registered results (PREREGISTRATION_map.md)", "",
               "| id | comparison | n | mean diff | 95% CI | p | p (Holm) | verdict |", "|---|---|---|---|---|---|---|---|"], []
    stats_rows = []
    for pid, key, a, b in PRIMARY:
        d = np.array([r[key.format(a)] - r[key.format(b)] for r in rows if key.format(a) in r and key.format(b) in r])
        if len(d) > 1 and d.std() > 0:
            p = stats.ttest_1samp(d, 0.0).pvalue
            half = stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / math.sqrt(len(d))
        else:
            p, half = (1.0 if len(d) < 2 else (0.0 if d.mean() != 0 else 1.0)), 0.0
        ps.append(p)
        stats_rows.append((pid, key.format(f"{a} - {b}"), len(d), d.mean() if len(d) else float("nan"), half, p))
    order, running, adj = np.argsort(ps), 0.0, [0.0] * len(ps)
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (len(ps) - rank) * ps[i]))
        adj[i] = running
    for (pid, comp, n, mean, half, p), pa in zip(stats_rows, adj):
        out.append(f"| {pid} | {comp} | {n} | {mean:+.4f} | ±{half:.4f} | {p:.3g} | {pa:.3g} | "
                   f"{'significant' if pa < 0.05 else 'not significant'} |")
    out += ["", "## All conditions (mean over seeds)", ""]
    keys = sorted({k for r in rows for k in r if k.endswith(("_acc", "_slope"))})
    for k in keys:
        out.append(f"- `{k}`: {np.mean([r[k] for r in rows if k in r]):.4f}")
    return "\n".join(out)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", type=int, default=10)
    ap.add_argument("--steps", type=int, default=6000, help="training steps per baseline per task (0: skip them)")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--out", default=os.path.join(HERE, "results", "map"))
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    rows, cfg = run(args.seeds, args.steps, args.device)
    json.dump({"args": vars(args), "map_config": cfg, "results": rows},
              open(os.path.join(args.out, f"map_{time.strftime('%Y%m%d-%H%M%S')}.json"), "w"), indent=1)
    # the "trained model" the chat interface loads: tuned settings only - it starts with no knowledge
    json.dump({"model": "NeuroMoE cognitive map", "map_config": cfg, "place_cells": {"per_dim": 48, "span": 24.0},
               "note": "no stored knowledge: it learns from what it is told"},
              open(os.path.join(args.out, "neuromoe_map_config.json"), "w"), indent=1)
    report = analyze(rows)
    open(os.path.join(args.out, "ANALYSIS_map.md"), "w", encoding="utf-8").write(report + "\n")
    print(report)
    print("saved to", args.out)

In [ ]:
%%writefile /kaggle/working/neuromoe/convo_bench.py
"""Conversational reasoning: the benchmark for PREREGISTRATION_convo.md.

Dialogues are generated, not scraped, so ground truth is exact: N people in an order, only adjacent pairs ever
stated, shuffled, some stated by pronoun, some stated backwards, distractor turns in between. Then questions
about pairs that were never put side by side.

    python convo_bench.py --seeds 10 --steps 4000 --out results/convo

Every system sees exactly the same turns in the same order. The brain sees each fact once and is never trained;
the transformer and GRU are trained on thousands of dialogues of this kind first.
"""
import argparse
import json
import math
import os
import random
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "chat"))

from brain.agent import Agent  # noqa: E402
from map_bench import batchify, train_net  # noqa: E402

NAMES = ["Alice", "Bob", "Cara", "Dan", "Eve", "Finn", "Gina", "Hugo", "Iris", "Jack", "Kira", "Liam",
         "Mona", "Nils", "Opal", "Pete"]
DISTRACTORS = ["That is interesting.", "I had a long day.", "Anyway.", "Tell me more later.",
               "The weather has been strange.", "I should get going soon."]
DIM = ("older", "younger")     # the dimension the dialogue is about, and its opposite


# ------------------------------------------------------------------------------------------------ episodes
def episode(n, noise, rng, pronoun_rate=0.35, distract_every=3):
    """-> (order, turns, questions). `turns` is what every system is given, in order.

    A turn is (kind, text, payload): kind 'fact' carries (hi, lo) as *stated* (possibly reversed), 'distract'
    carries nothing. Questions are asked afterwards and are never part of what is taught.
    """
    people = NAMES[:n]
    order = people[:]                       # index 0 is the oldest
    rng.shuffle(order)
    rank = {p: i for i, p in enumerate(order)}
    pairs = [(order[i], order[i + 1]) for i in range(n - 1)]   # (older, younger)
    rng.shuffle(pairs)
    flipped = set()
    for i in range(len(pairs)):
        if rng.random() < noise:
            pairs[i] = (pairs[i][1], pairs[i][0])
            flipped.add(i)

    turns, last_subject = [], None
    for i, (hi, lo) in enumerate(pairs):
        if hi == last_subject and rng.random() < pronoun_rate:
            text = f"She is older than {lo}."
            turns.append(("fact", text, {"stated": (hi, lo), "pronoun": True, "flipped": i in flipped}))
        else:
            text = f"{hi} is older than {lo}."
            turns.append(("fact", text, {"stated": (hi, lo), "pronoun": False, "flipped": i in flipped}))
        last_subject = hi
        if (i + 1) % distract_every == 0:
            turns.append(("distract", DISTRACTORS[rng.randrange(len(DISTRACTORS))], {}))

    stated_at = {}
    for t, (kind, _, payload) in enumerate(turns):
        if kind == "fact":
            stated_at[frozenset(payload["stated"])] = t
    bad_chain = _affected(order, rank, pairs, flipped)

    questions = []
    for i in range(n):
        for j in range(n):
            if i == j or abs(rank[people[i]] - rank[people[j]]) < 2:
                continue
            x, y = people[i], people[j]
            truth = rank[x] < rank[y]      # x is older
            span = _span(len(turns), stated_at, order, rank, x, y)
            questions.append({"x": x, "y": y, "truth": truth, "distance": abs(rank[x] - rank[y]),
                              "span": span, "contradicted": bool(bad_chain & _between(order, rank, x, y))})
    rng.shuffle(questions)
    return order, turns, questions


def _between(order, rank, x, y):
    lo, hi = sorted((rank[x], rank[y]))
    return {order[i] for i in range(lo, hi + 1)}


def _affected(order, rank, pairs, flipped):
    """People whose position in the chain rests on a fact that was stated backwards."""
    out = set()
    for i in flipped:
        out |= set(pairs[i])
    return out


def _span(n_turns, stated_at, order, rank, x, y):
    """How many turns ago the *most recent* fact on the chain between x and y was stated."""
    lo, hi = sorted((rank[x], rank[y]))
    links = [stated_at.get(frozenset((order[i], order[i + 1]))) for i in range(lo, hi)]
    links = [t for t in links if t is not None]
    return n_turns - max(links) if links else 0


def ask_text(q, pronoun=False):
    return f"Is {'she' if pronoun else q['x']} older than {q['y']}?"


def verdict(reply):
    """The yes/no a reply commits to, or None for an abstention."""
    low = reply.strip().lower()
    if low.startswith("yes"):
        return True
    if low.startswith("no") and not low.startswith("nothing"):
        return False
    return None


# -------------------------------------------------------------------------------------------------- systems
def run_brain(turns, questions, seed, ablate=(), sleep_at_end=True):
    """The whole loop, or the loop with one mechanism removed."""
    a = Agent(seed=seed, background=False)
    if "gate" in ablate:
        a.gate.pool.vigilance = 2.0          # nothing ever clears it: selection falls back to the reflexes
    if "wm" in ablate:
        a.wm.recall = lambda *args, **kw: None   # the slots are never read: nothing to resolve a pronoun against
    if "sleep" in ablate:
        a.sleep_every = 10 ** 9              # no consolidation, ever
    for kind, text, _ in turns:
        a.say(text)
    if sleep_at_end and "sleep" not in ablate:
        a.sleep()
    out = []
    for q in questions:
        if "map" in ablate:
            reply = a._do_recall({"intent": "recall"}, ask_text(q))["reply"]
        else:
            reply = a.say(ask_text(q))[-1]["reply"]
        out.append(verdict(reply))
    ref = []
    for q in questions[:12]:                 # C2: the same question, subject given as a pronoun
        a.say(f"Tell me about {q['x']}.")    # makes the referent unambiguous without stating any new fact
        reply = a.say(ask_text(q, pronoun=True))[-1]["reply"]
        ref.append((verdict(reply), q["truth"]))
    return out, ref, a


def run_lexical(turns, questions, rng):
    """Retrieval: answer from the earlier turn that shares most words with the question, and guess when the pair
    was never stated together. Guessing rather than abstaining is the stronger version of this baseline - it
    scores 0.5 where it has nothing instead of 0."""
    said = [(text, payload) for kind, text, payload in turns if kind == "fact"]
    out = []
    for q in questions:
        words = {q["x"].lower(), q["y"].lower()}
        best, score = None, 0
        for text, payload in said:
            overlap = len(words & set(text.lower().strip(".?").split()))
            if overlap > score:
                best, score = payload["stated"], overlap
        out.append(best[0] == q["x"] if best is not None and score >= 2 else rng.random() < 0.5)
    return out


def run_chain(turns, questions, n, rng):
    """Working-memory bindings only, answered by following links - memory without a map."""
    from map_bench import ChainMemory
    people = sorted({p for _, _, pay in turns if pay.get("stated") for p in pay["stated"]})
    idx = {p: i for i, p in enumerate(people)}
    c = ChainMemory(["older"])
    for kind, _, payload in turns:
        if kind == "fact":
            hi, lo = payload["stated"]
            c.observe(idx[lo], "older", idx[hi])
    out = []
    for q in questions:
        if q["x"] not in idx or q["y"] not in idx:
            out.append(rng.random() < 0.5)
            continue
        verdict_ = None
        # sign +1 follows "who is older than this" upward; reaching y that way means y is the older one
        for sign in (1, -1):
            z = idx[q["x"]]
            for _ in range(n):
                z = c.step(z, "older", sign)
                if z is None:
                    break
                if z == idx[q["y"]]:
                    verdict_ = sign == -1
                    break
            if verdict_ is not None:
                break
        out.append(verdict_ if verdict_ is not None else rng.random() < 0.5)
    return out


# ------------------------------------------------------------------------------------------ trained baselines
def vocab_size(n):
    return n + 5


def tokens(turns, questions, n, max_q=None, rng=None):
    OLDER, PRON, DISTRACT, QUERY = n, n + 1, n + 2, n + 3
    idx = {p: i for i, p in enumerate(NAMES[:n])}
    tok, tgt = [], []
    for kind, _, payload in turns:
        if kind == "fact":
            hi, lo = payload["stated"]
            tok += [PRON if payload["pronoun"] else idx[hi], OLDER, idx[lo]]
            tgt += [-1, -1, -1]
        else:
            tok += [DISTRACT]
            tgt += [-1]
    qs = questions if max_q is None else rng.sample(questions, min(max_q, len(questions)))
    for q in qs:
        tok += [QUERY, idx[q["x"]], idx[q["y"]]]
        tgt += [-1, -1, int(q["truth"])]
    return tok, tgt


class TinyTransformer(nn.Module):
    def __init__(self, vocab, d=128, layers=4, heads=4, max_len=1024, dropout=0.0):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.pos = nn.Parameter(0.02 * torch.randn(max_len, d))
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, activation="gelu", batch_first=True,
                                           norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.out = nn.Linear(d, 2)

    def forward(self, tok):
        L = tok.shape[1]
        mask = torch.triu(torch.full((L, L), float("-inf"), device=tok.device), 1)
        return self.out(self.blocks(self.emb(tok) + self.pos[:L], mask=mask, is_causal=True))


class GRUNet(nn.Module):
    def __init__(self, vocab, hidden=256, emb=64):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb)
        self.gru = nn.GRU(emb, hidden, batch_first=True)
        self.out = nn.Linear(hidden, 2)

    def forward(self, tok):
        return self.out(self.gru(self.emb(tok))[0])


def train_baselines(n, noise, steps, device, seed=0, lr=3e-4):
    """Trained on freshly generated dialogues of every noise level used in testing - never on a test episode."""
    rng = random.Random(90000 + seed)

    def make():
        _, turns, qs = episode(n, rng.choice((0.0, noise)), rng)
        return tokens(turns, qs, n, max_q=12, rng=rng)

    nets = {}
    for name, net in (("transformer", TinyTransformer(vocab_size(n))), ("gru", GRUNet(vocab_size(n)))):
        torch.manual_seed(seed)
        nets[name] = train_net(net.to(device), make, steps, lr, device)
    return nets


@torch.no_grad()
def net_verdicts(net, turns, questions, n, device, chunk=12):
    out = []
    for i in range(0, len(questions), chunk):
        tok, tgt = tokens(turns, questions[i:i + chunk], n)
        t, g = batchify([(tok, tgt)], device)
        pred = net(t).argmax(-1)[0]
        out += [bool(v) for v in pred[g[0] >= 0].cpu().tolist()]
    return out


# ------------------------------------------------------------------------------------------------------ bench
def score(verdicts, questions, pick=lambda q: True):
    hits = [v == q["truth"] for v, q in zip(verdicts, questions) if pick(q)]
    absts = [v is None for v, q in zip(verdicts, questions) if pick(q)]
    return (float(np.mean(hits)) if hits else float("nan"),
            float(np.mean(absts)) if absts else 0.0)


def run(seeds, steps, n, noise, out_dir, device):
    os.makedirs(out_dir, exist_ok=True)
    nets = train_baselines(n, noise, steps, device) if steps else {}
    rows = []
    for s in seeds:
        rng = random.Random(s)
        t0 = time.time()
        order, turns, questions = episode(n, noise, rng)
        row = {"seed": s, "n": n, "noise": noise, "turns": len(turns), "questions": len(questions)}

        v, ref, agent = run_brain(turns, questions, s)
        row["acc_brain"], row["abst_brain"] = score(v, questions)
        row["c3_brain"], _ = score(v, questions, lambda q: q["span"] >= 8)
        row["c4_brain"], _ = score(v, questions, lambda q: q["contradicted"])
        row["ref_brain"] = float(np.mean([a == t for a, t in ref])) if ref else float("nan")
        row["gate_learned"] = float(np.mean([t["via"] == "learned" for t in agent.trace]))
        row["footprint"] = agent.footprint()

        for name, ablate in (("nowm", ("wm",)), ("nomap", ("map",)), ("nogate", ("gate",)), ("nosleep", ("sleep",))):
            va, ra, _ = run_brain(turns, questions, s, ablate=ablate)
            row[f"acc_{name}"], row[f"abst_{name}"] = score(va, questions)
            row[f"ref_{name}"] = float(np.mean([a == t for a, t in ra])) if ra else float("nan")

        row["acc_lexical"], row["abst_lexical"] = score(run_lexical(turns, questions, rng), questions)
        row["acc_chain"], row["abst_chain"] = score(run_chain(turns, questions, n, rng), questions)
        for name, net in nets.items():
            row[f"acc_{name}"], row[f"abst_{name}"] = score(net_verdicts(net, turns, questions, n, device),
                                                            questions)
        row["seconds"] = round(time.time() - t0, 1)
        rows.append(row)
        print(f"seed {s}: brain {row['acc_brain']:.3f} | lexical {row['acc_lexical']:.3f} | "
              f"chain {row['acc_chain']:.3f} | " +
              " | ".join(f"{k} {row[f'acc_{k}']:.3f}" for k in nets) +
              f" | ref {row['ref_brain']:.3f} vs nowm {row['ref_nowm']:.3f} ({row['seconds']}s)", flush=True)
    stamp = time.strftime("%Y%m%d-%H%M%S")
    with open(os.path.join(out_dir, f"convo_{stamp}.json"), "w", encoding="utf-8") as f:
        json.dump({"rows": rows, "n": n, "noise": noise, "steps": steps}, f, indent=1)
    report = analyze(rows)
    with open(os.path.join(out_dir, "ANALYSIS_convo.md"), "w", encoding="utf-8") as f:
        f.write(report + "\n")
    print(report)
    return rows


PRIMARY = [("P1", "acc_{}", "brain", "lexical"), ("P2", "acc_{}", "brain", "chain"),
           ("P3", "acc_{}", "brain", "transformer"), ("P4", "ref_{}", "brain", "nowm")]


def analyze(rows):
    from scipy import stats
    out = ["# Conversational reasoning: pre-registered results (PREREGISTRATION_convo.md)", "",
           "| id | comparison | n | mean diff | 95% CI | p | p (Holm) | verdict |",
           "|---|---|---|---|---|---|---|---|"]
    ps, kept = [], []
    for pid, key, a, b in PRIMARY:
        d = np.array([r[key.format(a)] - r[key.format(b)] for r in rows
                      if key.format(a) in r and key.format(b) in r and not math.isnan(r[key.format(b)])])
        if len(d) > 1 and d.std() > 0:
            p = float(stats.ttest_1samp(d, 0.0).pvalue)
            half = float(stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / math.sqrt(len(d)))
        else:
            p, half = (1.0 if len(d) < 2 else 0.0), 0.0
        ps.append(p)
        kept.append((pid, a, b, d, half))
    order = np.argsort(ps)
    holm = [0.0] * len(ps)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, ps[i] * (len(ps) - rank))
        holm[i] = min(1.0, running)
    for (pid, a, b, d, half), p, h in zip(kept, ps, holm):
        verdict_ = "supported" if h < 0.05 and d.mean() > 0 else ("against" if h < 0.05 else "not significant")
        out.append(f"| {pid} | {a} - {b} | {len(d)} | {d.mean():+.3f} | +-{half:.3f} | {p:.4f} | {h:.4f} | "
                   f"{verdict_} |")
    out += ["", "## Every system", "", "| system | C1 inference | abstention |", "|---|---|---|"]
    for name in ("brain", "nowm", "nomap", "nogate", "nosleep", "lexical", "chain", "transformer", "gru"):
        key, akey = f"acc_{name}", f"abst_{name}"
        vals = [r[key] for r in rows if key in r and not math.isnan(r[key])]
        absts = [r[akey] for r in rows if akey in r]
        if vals:
            out.append(f"| {name} | {np.mean(vals):.3f} | {np.mean(absts):.3f} |")
    out += ["", "## Exploratory", ""]
    for key, label in (("ref_brain", "C2 reference (brain)"), ("ref_nowm", "C2 reference (no working memory)"),
                       ("c3_brain", "C3 span >= 8 turns"), ("c4_brain", "C4 contradicted chains"),
                       ("gate_learned", "turns selected by an expert, not a reflex"),
                       ("footprint", "synapses after the dialogue")):
        vals = [r[key] for r in rows if key in r and not math.isnan(r[key])]
        if vals:
            out.append(f"- **{label}**: {np.mean(vals):.3f}")
    return "\n".join(out)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", type=int, default=10)
    ap.add_argument("--n", type=int, default=10)
    ap.add_argument("--noise", type=float, default=0.1)
    ap.add_argument("--steps", type=int, default=4000)
    ap.add_argument("--out", default=os.path.join(ROOT, "results", "convo"))
    args = ap.parse_args()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device {device}, {args.seeds} seeds, N={args.n}, noise={args.noise}, baseline steps={args.steps}",
          flush=True)
    run(range(args.seeds), args.steps, args.n, args.noise, args.out, device)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/neuromoe/compare_runs.py
"""Compare two conversational-reasoning runs seed by seed.

    python compare_runs.py results/convo/convo_A.json results/convo/convo_B.json --labels v1 v2

Both runs see identical dialogues for identical seeds - the generator is seeded and the architecture change does
not touch it - so every metric can be paired, which is the only honest way to attribute a difference to the
change rather than to the episodes.
"""
import argparse
import json
import math

import numpy as np

METRICS = [("acc_brain", "C1 inference"), ("ref_brain", "C2 reference"), ("c3_brain", "C3 span >= 8 turns"),
           ("c4_brain", "C4 contradicted chains"), ("abst_brain", "abstention"),
           ("gate_learned", "turns chosen by an expert"), ("acc_nowm", "ablation: no working memory"),
           ("acc_nogate", "ablation: no gate"), ("acc_nosleep", "ablation: no sleep"),
           ("acc_lexical", "control: retrieval"), ("acc_chain", "control: link-following"),
           ("acc_transformer", "control: transformer"), ("acc_gru", "control: GRU"),
           ("footprint", "synapses"), ("seconds", "seconds per episode")]


def load(path):
    with open(path, encoding="utf-8") as f:
        return {r["seed"]: r for r in json.load(f)["rows"]}


def paired(a, b, key):
    seeds = sorted(set(a) & set(b))
    xs = [(a[s][key], b[s][key]) for s in seeds
          if key in a[s] and key in b[s] and not math.isnan(a[s][key]) and not math.isnan(b[s][key])]
    return np.array([x for x, _ in xs]), np.array([y for _, y in xs])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("runs", nargs=2)
    ap.add_argument("--labels", nargs=2, default=["A", "B"])
    ap.add_argument("--out")
    args = ap.parse_args()
    from scipy import stats

    A, B = load(args.runs[0]), load(args.runs[1])
    la, lb = args.labels
    lines = [f"# {la} vs {lb}, paired over {len(set(A) & set(B))} seeds", "",
             f"| metric | {la} | {lb} | difference | p (paired) |", "|---|---|---|---|---|"]
    for key, label in METRICS:
        x, y = paired(A, B, key)
        if not len(x):
            continue
        d = y - x
        p = float(stats.ttest_rel(y, x).pvalue) if len(d) > 1 and d.std() > 0 else 1.0
        fmt = (lambda v: f"{v:,.0f}") if key in ("footprint",) else (lambda v: f"{v:.3f}")
        lines.append(f"| {label} | {fmt(x.mean())} | {fmt(y.mean())} | {d.mean():+.3f} | {p:.4f} |")
    report = "\n".join(lines)
    print(report)
    if args.out:
        with open(args.out, "w", encoding="utf-8") as f:
            f.write(report + "\n")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/neuromoe/chat/language.py
"""Controlled English for NeuroMoE's cognitive map. A hand-written parser - no language model - that turns a fixed set
of sentence patterns into facts ("teach") and questions ("ask"). Anything else is refused rather than guessed at.

Facts      Alice is taller than Bob.          The cafe is north of the park.      Dan is 2 blocks east of Eve.
           Bob is shorter than Alice.         The bank is north-east of the park. Cara is more careful than Dan.
Questions  Is Alice taller than Cara?         Who is taller, Alice or Dan?        Who is the tallest?
           Rank everyone by height.           Who is between Alice and Dan?       How much taller is Alice than Dan?
           What is 2 blocks north of the park?  Is the bank north of the cafe?    Where is the cafe relative to the bank?
           Tell me about Alice.
Commands   sleep | what do you know | forget everything | help
"""
import re

NUMBERS = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9,
           "ten": 10, "a": 1, "an": 1}
NUM = r"(?:\d+|one|two|three|four|five|six|seven|eight|nine|ten|a|an)"

# comparative -> (axis, sign). The axis is a dimension of its own; sign says which way along it the word points.
COMPARATIVES = {
    "taller": ("height", 1), "shorter": ("height", -1), "older": ("age", 1), "younger": ("age", -1),
    "bigger": ("size", 1), "larger": ("size", 1), "smaller": ("size", -1), "heavier": ("weight", 1),
    "lighter": ("weight", -1), "faster": ("speed", 1), "slower": ("speed", -1), "richer": ("wealth", 1),
    "poorer": ("wealth", -1), "stronger": ("strength", 1), "weaker": ("strength", -1), "smarter": ("smarts", 1),
    "better": ("quality", 1), "worse": ("quality", -1), "hotter": ("heat", 1), "colder": ("heat", -1),
    "warmer": ("heat", 1), "cooler": ("heat", -1), "louder": ("loudness", 1), "quieter": ("loudness", -1),
    "happier": ("happiness", 1), "sadder": ("happiness", -1), "cheaper": ("price", -1), "longer": ("length", 1),
}
SUPERLATIVE_EXCEPTIONS = {"best": "better", "worst": "worse"}
NOUNS = {"height": "height", "age": "age", "size": "size", "weight": "weight", "speed": "speed", "wealth": "wealth",
         "strength": "strength", "price": "price", "length": "length", "heat": "heat", "temperature": "heat",
         "quality": "quality", "loudness": "loudness", "happiness": "happiness"}

# spatial words -> (axis, sign). Every spatial axis lives in one shared space, so routes may mix them.
# Left/right are read as on a map (right = east); in front/behind and above/below get axes of their own.
DIRECTIONS = {"north": ("north", 1), "south": ("north", -1), "east": ("east", 1), "west": ("east", -1),
              "above": ("up", 1), "below": ("up", -1), "under": ("up", -1), "over": ("up", 1),
              "left": ("east", -1), "right": ("east", 1), "behind": ("front", -1), "in front": ("front", 1)}
SPATIAL_AXES = ("north", "east", "up", "front")
SAY_AXIS = {"north": ("north", "south"), "east": ("east", "west"), "up": ("above", "below"),
            "front": ("in front", "behind")}


def space_of(axis):
    return "place" if axis in SPATIAL_AXES else axis


def positive_word(axis):
    """The word for one step up an axis: 'height' -> 'taller', 'careful' -> 'more careful'."""
    for word, (a, s) in COMPARATIVES.items():
        if a == axis and s == 1:
            return word
    return axis if axis.endswith("er") else "more " + axis


def clean(text):
    text = text.strip().rstrip("?.!;").strip()
    return re.sub(r"\s+", " ", text)


def sentences(text):
    """Split a message into sentences, so a whole story can be taught in one go."""
    return [s.strip() for s in re.split(r"(?<=[.!?;])\s+|\n+", text) if clean(s)]


POSSESSIVE = r"(?:the|a|an|my|your|his|her|our|their|its)"


def name(text):
    """'the Cafe' -> ('cafe', 'the Cafe'): a lookup key and how to say it.

    "my dog Rex" is Rex: when a phrase introduces something by category and then names it, the name is what the
    speaker will use next time, so that is what it gets filed under.
    """
    shown = text.strip().strip(",").strip()
    key = re.sub(r"^" + POSSESSIVE + r"\s+", "", shown.lower()).strip()
    words = shown.split()
    # "my dog Rex" -> Rex, but "the Cafe" stays as it is: it takes a category word in between for the last word
    # to be a name being introduced rather than part of the thing's own name
    if len(words) > 2 and words[-1][:1].isupper() and not words[-2][:1].isupper():
        key, shown = words[-1].lower(), words[-1]
    return key, shown


def comparative(word):
    """Known comparatives, plus any unknown '-er' word as a dimension of its own (so 'nicer' just works)."""
    w = word.lower()
    if w in COMPARATIVES:
        return COMPARATIVES[w]
    if len(w) > 4 and w.endswith("er") and w not in ("other", "never", "under", "over", "either", "rather"):
        return (w, 1)
    return None


def superlative(word):
    w = word.lower()
    if w in SUPERLATIVE_EXCEPTIONS:
        return comparative(SUPERLATIVE_EXCEPTIONS[w])
    if len(w) > 4 and w.endswith("est"):
        stem = w[:-3]
        for guess in (stem + "er", stem[:-1] + "ier" if stem.endswith("i") else None, stem + "r"):
            if guess and guess in COMPARATIVES:
                return COMPARATIVES[guess]
        return comparative(stem + "er")
    return None


def count(text):
    if not text:
        return 1
    text = text.strip().lower()
    return int(text) if text.isdigit() else NUMBERS.get(text)


COMPASS = r"(north|south|east|west)(?:[- ]?(north|south|east|west))?"
_ONE = r"(?:" + NUM + r"\s+)?(?:(?:blocks?|steps?|miles?|km|meters?|units?|streets?)\s+)?(?:north|south|east|west)(?:[- ]?(?:north|south|east|west))?"
MOVES = r"(" + _ONE + r"(?:(?:\s+and\s+|,\s*)" + _ONE + r")*)"


def moves(text):
    """'2 blocks north-east' / 'north' / '3 north and 1 east' -> [(axis, count), ...] or None."""
    total = {}
    for part in re.split(r"\s+and\s+|,\s*", text.strip().lower()):
        m = re.fullmatch(r"(?:(" + NUM + r")\s+)?(?:(?:blocks?|steps?|miles?|km|meters?|units?|streets?)\s+)?"
                         + COMPASS, part.strip())
        if not m:
            return None
        n = count(m.group(1))
        for word in filter(None, (m.group(2), m.group(3))):
            axis, sign = DIRECTIONS[word]
            total[axis] = total.get(axis, 0) + sign * n
    return [(a, c) for a, c in total.items() if c] or None


def _cmp(word):
    """A comparative phrase ('taller', 'more careful', 'less careful') -> (axis, sign, word) or None."""
    m = re.fullmatch(r"(more|less) (\w+)", word, re.I)
    if m:
        return m.group(2).lower(), 1 if m.group(1).lower() == "more" else -1, word.lower()
    c = comparative(word)
    return (c[0], c[1], word.lower()) if c else None


SPATIAL_WORDS = "|".join(sorted(DIRECTIONS, key=len, reverse=True))
HEDGE = r"(?:a bit |much |slightly |a lot |way |far |even )?"


def parse(text):
    """-> dict with 'kind' in {fact, ask, command, unknown} and whatever that kind needs."""
    t = clean(text)
    low = t.lower()

    if low in ("sleep", "consolidate", "go to sleep", "rest", "dream"):
        return {"kind": "command", "do": "sleep"}
    if low in ("forget everything", "reset", "clear", "start over"):
        return {"kind": "command", "do": "reset"}
    if low in ("what do you know", "show the map", "show map", "status", "what do you remember", "summary"):
        return {"kind": "command", "do": "summary"}
    if low in ("help", "what can i say", "examples", "what can you do"):
        return {"kind": "command", "do": "help"}

    # questions -------------------------------------------------------------------------------------------
    m = re.fullmatch(r"is (.+?) " + HEDGE + r"((?:more|less) \w+|\w+) than (.+)", t, re.I)
    if m and _cmp(m.group(2)):
        axis, sign, word = _cmp(m.group(2))
        return {"kind": "ask", "q": "compare", "a": name(m.group(1)), "b": name(m.group(3)), "axis": axis,
                "sign": sign, "word": word}
    m = re.fullmatch(r"is (.+?) (?:directly |exactly )?" + MOVES + r" of (.+)", t, re.I)
    if m and moves(m.group(2)):
        exact = bool(re.match(r"\s*" + NUM + r"\s", m.group(2), re.I) and not re.match(r"\s*an?\s", m.group(2), re.I))
        return {"kind": "ask", "q": "direction", "a": name(m.group(1)), "b": name(m.group(3)),
                "moves": moves(m.group(2)), "exact": exact, "word": m.group(2).lower()}
    m = re.fullmatch(r"is (.+?) (?:to the |directly |just )?(" + SPATIAL_WORDS + r")(?: of)? (.+)", t, re.I)
    if m:
        axis, sign = DIRECTIONS[m.group(2).lower()]
        return {"kind": "ask", "q": "direction", "a": name(m.group(1)), "b": name(m.group(3)),
                "moves": [(axis, sign)], "exact": False, "word": m.group(2).lower()}
    m = re.fullmatch(r"(?:who|which|what)(?: one)? is ((?:more|less) \w+|\w+),? (.+?),? or (.+)", t, re.I)
    if m and _cmp(m.group(1)):
        axis, sign, word = _cmp(m.group(1))
        return {"kind": "ask", "q": "which", "a": name(m.group(2)), "b": name(m.group(3)), "axis": axis,
                "sign": sign, "word": word}
    m = re.fullmatch(r"is (.+?) the (most|least) (\w+)(?: one)?", t, re.I)
    if m:
        return {"kind": "ask", "q": "extreme", "a": name(m.group(1)), "axis": m.group(3).lower(),
                "sign": 1 if m.group(2).lower() == "most" else -1, "word": f"{m.group(2)} {m.group(3)}".lower()}
    m = re.fullmatch(r"is (.+?) the (\w+)(?: one)?", t, re.I)
    if m and superlative(m.group(2)):
        axis, sign = superlative(m.group(2))
        return {"kind": "ask", "q": "extreme", "a": name(m.group(1)), "axis": axis, "sign": sign,
                "word": m.group(2).lower()}
    m = re.fullmatch(r"(?:who|which|what)(?: one)? is the (most|least) (\w+)(?: one)?", t, re.I)
    if m:
        return {"kind": "ask", "q": "extreme", "axis": m.group(2).lower(),
                "sign": 1 if m.group(1).lower() == "most" else -1, "word": f"{m.group(1)} {m.group(2)}".lower()}
    m = re.fullmatch(r"(?:who|which|what)(?: one)? is the (\w+)(?: one)?", t, re.I)
    if m and superlative(m.group(1)):
        axis, sign = superlative(m.group(1))
        return {"kind": "ask", "q": "extreme", "axis": axis, "sign": sign, "word": m.group(1).lower()}
    m = re.fullmatch(r"(?:rank|order|sort|list)(?: everyone| everything| them| all| the \w+)?(?: by| from)? (.+)", t,
                     re.I)
    if m:
        what = re.sub(r"\s+to\s+\w+$", "", m.group(1).lower().strip())  # "tallest to shortest" -> "tallest"
        found = None
        if what in NOUNS:
            found = (NOUNS[what], 1)
        elif re.fullmatch(r"(most|least) (\w+)", what):
            mm = re.fullmatch(r"(most|least) (\w+)", what)
            found = (mm.group(2), 1 if mm.group(1) == "most" else -1)
        else:
            found = superlative(what) or comparative(what) or ((what, 1) if re.fullmatch(r"\w+", what) else None)
        if found:
            return {"kind": "ask", "q": "rank", "axis": found[0], "sign": found[1]}
    m = re.fullmatch(r"(?:who|what)(?: is| are|'s)? between (.+?) and (.+)", t, re.I)
    if m:
        return {"kind": "ask", "q": "between", "a": name(m.group(1)), "b": name(m.group(2))}
    m = re.fullmatch(r"how (?:much|many \w+|far) ((?:more|less) \w+|\w+) is (.+?) than (.+)", t, re.I)
    if m and _cmp(m.group(1)):
        axis, sign, word = _cmp(m.group(1))
        return {"kind": "ask", "q": "howmuch", "a": name(m.group(2)), "b": name(m.group(3)), "axis": axis,
                "sign": sign, "word": word}
    m = re.fullmatch(r"(?:what|who|which \w+)(?: is|'s| lies)? " + MOVES + r" of (.+)", t, re.I)
    if m and moves(m.group(1)):
        return {"kind": "ask", "q": "at", "a": name(m.group(2)), "moves": moves(m.group(1)), "word": m.group(1).lower()}
    m = re.fullmatch(r"where is (.+?)(?: relative to| compared to| from| in relation to) (.+)", t, re.I)
    if m:
        return {"kind": "ask", "q": "where", "a": name(m.group(1)), "b": name(m.group(2))}
    m = re.fullmatch(r"how far is (.+?) from (.+)", t, re.I)
    if m:
        return {"kind": "ask", "q": "where", "a": name(m.group(1)), "b": name(m.group(2))}
    m = re.fullmatch(r"where is (.+)", t, re.I)
    if m:
        return {"kind": "ask", "q": "about", "a": name(m.group(1))}
    m = re.fullmatch(r"(?:what do you know about|tell me about|describe|who is|what is) (.+)", t, re.I)
    if m:
        return {"kind": "ask", "q": "about", "a": name(m.group(1))}

    # facts -----------------------------------------------------------------------------------------------
    m = re.fullmatch(r"(.+?) (?:is|are|was|were) " + HEDGE + r"((?:more|less) \w+|\w+) than (.+)", t, re.I)
    if m and _cmp(m.group(2)):
        axis, sign, word = _cmp(m.group(2))
        return {"kind": "fact", "a": name(m.group(1)), "b": name(m.group(3)), "moves": [(axis, sign)], "word": word}
    m = re.fullmatch(r"(.+?) (?:is|are) (?:directly |just )?" + MOVES + r" of (.+)", t, re.I)
    if m and moves(m.group(2)):
        return {"kind": "fact", "a": name(m.group(1)), "b": name(m.group(3)), "moves": moves(m.group(2)),
                "word": m.group(2).lower()}
    m = re.fullmatch(r"(.+?) (?:is|are) (?:directly |just |to the )?(" + SPATIAL_WORDS + r")(?: of)? (.+)", t, re.I)
    if m:
        axis, sign = DIRECTIONS[m.group(2).lower()]
        return {"kind": "fact", "a": name(m.group(1)), "b": name(m.group(3)), "moves": [(axis, sign)],
                "word": m.group(2).lower()}

    return {"kind": "unknown"}

In [ ]:
%%writefile /kaggle/working/neuromoe/chat/mapsystem.py
"""The conversational wrapper around NeuroMoE's cognitive map. It holds one map per kind of relation - one for
places (north/east/up/front share a space, so routes can mix them), and one per comparative dimension (height, age,
...), because being taller says nothing about being older. Every answer is read off the maps; this file only turns
parsed sentences into observe / sleep / steps / nearest calls and turns the numbers back into words.

Nothing is pre-loaded: the maps start empty and learn one-shot from what they are told. Settings are the ones tuned
on validation episodes in the Kaggle run (results/kaggle_map/results/map/neuromoe_map_config.json).
"""
import json
import os
import re
import sys

import torch

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, ROOT)
from cogmap import CognitiveMap, PlaceCells  # noqa: E402

import language as L  # noqa: E402

CONFIG = os.path.join(ROOT, "results", "kaggle_map", "results", "map", "neuromoe_map_config.json")
PLACE_DIM, LINE_DIM = 4, 2  # places: north, east, up, front; each comparative dimension: its axis plus slack

HELP = """I learn from what you tell me - nothing is pre-loaded. I understand a small, fixed kind of English:

Teach me:  "Alice is taller than Bob."  "Cara is more careful than Dan."
           "The cafe is north of the park."  "The bank is 2 blocks east of the cafe."  "The lamp is left of the bed."
Ask me:    "Is Alice taller than Dan?"  "Who is taller, Bob or Dan?"  "Who is the tallest?"  "Rank everyone by height."
           "Who is between Alice and Dan?"  "How much taller is Alice than Dan?"
           "What is 2 blocks north of the park?"  "Is the bank north-east of the park?"  "Where is the bank from the park?"
           "Tell me about Alice."
Commands:  "sleep" (consolidate), "what do you know", "forget everything".

You can paste several sentences at once. I only ever answer from what you have told me; if two things were never
linked by any chain of facts, I say I don't know."""


# The settings chosen on validation episodes in the pseudospatial Kaggle run, kept here so the map is the same
# one that was measured even on a machine without that run's output folder. results/kaggle_map/.../
# neuromoe_map_config.json is the same numbers; if it is present it wins, so a re-tune takes effect.
TUNED = {"model": "NeuroMoE cognitive map",
         "map_config": {"lr": 0.3, "rel_lr": 0.0, "replay": 40, "robust": 0.3, "sleep": 300, "tol": 0.3},
         "place_cells": {"per_dim": 48, "span": 24.0}}


def load_config():
    if not os.path.exists(CONFIG):
        return TUNED
    with open(CONFIG, encoding="utf-8") as f:
        return json.load(f)


class Brain:
    def __init__(self, config=None, seed=0):
        self.config = config or load_config()
        self.seed = seed
        self.reset()

    # ------------------------------------------------------------------ state
    def reset(self):
        self.maps = {}        # space -> CognitiveMap
        self.shown = {}       # key -> how the user wrote it
        self.words = {}       # comparative axis -> the word the user used for "more"
        self.log = []         # every sentence taught, in order (the model's whole knowledge, replayable)
        self.dirty = set()    # spaces with new facts since the last sleep

    def _map(self, space):
        if space not in self.maps:
            c = self.config["map_config"]
            dim = PLACE_DIM if space == "place" else LINE_DIM
            pc = self.config.get("place_cells", {})
            place = PlaceCells(dim=dim, per_dim=pc.get("per_dim", 48), span=pc.get("span", 24.0),
                               seed=self.seed + len(self.maps))
            self.maps[space] = CognitiveMap(dim=dim, lr=c["lr"], rel_lr=c["rel_lr"], replay=c["replay"],
                                            robust=c["robust"], place=place, seed=self.seed + len(self.maps))
        return self.maps[space]

    def _say(self, key):
        return self.shown.get(key) or getattr(self, "_asked", {}).get(key, key)

    def _connected(self, space, a, b):
        m = self.maps.get(space)
        return bool(m and m.knows(a) and m.knows(b) and m.island[m.index[a]] == m.island[m.index[b]])

    def _settle(self, space):
        """Sleep before answering if new facts arrived: every episode replayed, the map relaxes as a whole."""
        if space in self.dirty and space in self.maps:
            self.maps[space].sleep(self.config["map_config"]["sleep"], tolerance=self.config["map_config"]["tol"])
            self.dirty.discard(space)
            return True
        return False

    def _offset(self, space, a, b):
        """Steps from a to b along every relation of the space, e.g. {'north': 2.0, 'east': -1.0}."""
        m = self.maps[space]
        return {r: m.steps(a, r, b) for r in m.rel}

    # ------------------------------------------------------------------ dialogue
    def say(self, text):
        """-> list of {sentence, parsed, reply, kind}. Several sentences can arrive in one message."""
        out = []
        for s in L.sentences(text) or [text]:
            p = L.parse(s)
            try:
                reply = self._handle(p, s)
            except Exception as e:  # never let one odd sentence kill the conversation
                reply = f"(internal error: {type(e).__name__}: {e})"
            reply = reply[:1].upper() + reply[1:]
            out.append({"sentence": s, "parsed": describe(p), "reply": reply, "kind": p["kind"]})
        return out

    def _handle(self, p, sentence):
        k = p["kind"]
        if k == "command":
            return getattr(self, "_cmd_" + p["do"])()
        if k == "fact":
            self.log.append(sentence)
            return self._teach(p)
        if k == "ask":
            self._asked = {x[0]: article(x[1]) for x in (p.get("a"), p.get("b")) if x}
            return getattr(self, "_ask_" + p["q"])(p)
        return ("I didn't understand that. I only follow a small set of sentence shapes - facts like "
                "\"A is taller than B\" or \"A is north of B\", and questions about them. Type \"help\" for the list.")

    # ------------------------------------------------------------------ teaching
    def _teach(self, p):
        (ka, sa), (kb, sb) = p["a"], p["b"]
        if ka == kb:
            return "That relates something to itself, so I can't place it."
        self.shown.setdefault(ka, article(sa))
        self.shown.setdefault(kb, article(sb))
        spaces = {L.space_of(axis) for axis, _ in p["moves"]}
        space = spaces.pop()
        moves = tuple(p["moves"])
        if space != "place":
            axis, sign = moves[0]
            if sign == 1 or axis not in self.words:
                self.words.setdefault(axis, p["word"] if sign == 1 else L.positive_word(axis))
        m = self._map(space)
        note = ""
        if self._connected(space, kb, ka):  # already inferable: does the new fact agree with what I inferred?
            self._settle(space)
            predicted = self._offset(space, kb, ka)
            for axis, c in moves:
                have = predicted.get(axis, 0.0)
                if c * have < -0.5:
                    note = (f" That contradicts what I had inferred, so I'll weigh it against the other facts "
                            f"(one fact against many gets outvoted when I sleep).")
                    break
            else:
                note = " I could already infer that - it fits."
        new = [x for x in (ka, kb) if not m.knows(x)]
        joins = not new and not self._connected(space, ka, kb)
        m.observe_moves(kb, moves, ka)  # a = b + moves
        if joins:
            self.dirty.add(space)
            size = sum(self._connected(space, ka, n) for n in m.names)
            return (f"Got it - that links two separate groups, so I shifted one to fit and merged them. "
                    f"{size} things are now on one map, and I can compare any two of them.")
        self.dirty.add(space)
        if len(new) == 2:
            where = "a new, separate group - I'll link it to the rest once a fact connects them"
            if not any(m.knows(n) for n in m.names if n not in (ka, kb)):
                where = "the start of a new map" if space == "place" else f"the start of my {self._dim(axis)} scale"
            return f"Got it. {sa} and {sb} are {where}.{note}"
        if len(new) == 1:
            return f"Got it - I placed {self._say(new[0])} from that one fact.{note}"
        return f"Got it.{note}"

    def _dim(self, axis):
        return {"smarts": "smartness"}.get(axis, axis)

    # ------------------------------------------------------------------ questions
    def _unknown(self, *keys, space=None):
        missing = [self._say(k) for k in keys if not any(m.knows(k) for m in self.maps.values())]
        if missing:
            return f"I don't know anything about {' or '.join(missing)} yet."
        return None

    def _link_check(self, space, a, b, what):
        """None if a and b can be compared in this space, otherwise the honest reason why not."""
        m = self.maps.get(space)
        if m is None:
            return f"I don't know - you haven't told me anything about {what} yet."
        outside = [self._say(x) for x in (a, b) if not m.knows(x)]
        if outside:
            who = " or ".join(outside)
            return (f"I don't know - you haven't told me where {who} is." if space == "place" else
                    f"I don't know - you haven't told me how {who} compares by {what}.")
        return None if self._connected(space, a, b) else self._not_linked(a, b, what)

    def _not_linked(self, a, b, what):
        how = "in space" if what == "where things are" else f"by {what}"
        return (f"I don't know - nothing you've told me links {self._say(a)} and {self._say(b)} {how}. "
                f"They're on separate islands of my map.")

    def _confidence(self, s):
        s = abs(s)
        return "" if s >= 1.5 else " (I think so - they're close)" if s >= 0.5 else ""

    def _ask_compare(self, p):
        (ka, _), (kb, _) = p["a"], p["b"]
        axis, sign, word = p["axis"], p["sign"], p["word"]
        miss = self._unknown(ka, kb)
        if miss:
            return miss
        space = L.space_of(axis)
        why = self._link_check(space, ka, kb, self._dim(axis))
        if why:
            return why
        self._settle(space)
        s = self.maps[space].steps(kb, axis, ka) * sign  # >0: a is `word` than b
        if abs(s) < 0.5:
            return f"I can't tell - {self._say(ka)} and {self._say(kb)} land in about the same place ({s:+.1f} steps)."
        if s > 0:
            return f"Yes, {self._say(ka)} is {word} than {self._say(kb)}.{self._confidence(s)}"
        return f"No - {self._say(kb)} is {word} than {self._say(ka)}.{self._confidence(s)}"

    def _ask_which(self, p):
        (ka, _), (kb, _) = p["a"], p["b"]
        axis, sign, word = p["axis"], p["sign"], p["word"]
        miss = self._unknown(ka, kb)
        if miss:
            return miss
        space = L.space_of(axis)
        why = self._link_check(space, ka, kb, self._dim(axis))
        if why:
            return why
        self._settle(space)
        s = self.maps[space].steps(kb, axis, ka) * sign
        if abs(s) < 0.5:
            return f"I can't tell them apart ({s:+.1f} steps)."
        win, lose = (ka, kb) if s > 0 else (kb, ka)
        return f"{self._say(win)} is {word} than {self._say(lose)}.{self._confidence(s)}"

    def _groups(self, space):
        m = self.maps[space]
        groups = {}
        for n in m.names:
            if m.knows(n):
                groups.setdefault(m.island[m.index[n]], []).append(n)
        return list(groups.values())

    def _ask_extreme(self, p):
        axis, sign, word = p["axis"], p["sign"], p["word"]
        space = L.space_of(axis)
        if space not in self.maps or axis not in self.maps[space].rel:
            return f"You haven't told me anything about who is {L.positive_word(axis)} yet."
        self._settle(space)
        m = self.maps[space]
        tops = []
        for g in self._groups(space):
            order = sorted(g, key=lambda n: -sign * float(m.pos[m.index[n]] @ m.rel[axis]))
            gap = abs(m.steps(order[1], axis, order[0])) if len(order) > 1 else 9
            tops.append((order[0], gap, order))
        asked = p["a"][0] if p.get("a") else None     # "is Rex the oldest?" - the same question, answered yes or no
        if len(tops) == 1:
            top, gap, order = tops[0]
            hedge = "" if gap >= 0.5 else f" - but {self._say(order[1])} is about level, so I'm not sure"
            if asked:
                if asked not in order:
                    return f"I don't know - you haven't told me how {self._say(asked)} compares by {self._dim(axis)}."
                if asked == top:
                    return f"Yes, {self._say(asked)} is the {word}{hedge}."
                return (f"No - {self._say(top)} is, and {self._say(asked)} is "
                        f"{ordinal(order.index(asked) + 1)} of {len(order)}.")
            return f"{self._say(top)} is the {word}{hedge}."
        if asked:
            for top, gap, order in tops:
                if asked in order:
                    rank = ordinal(order.index(asked) + 1)
                    return (f"Of the {len(order)} I can compare with {self._say(asked)}, it is {rank}"
                            + ("" if asked == top else f" - {self._say(top)} is the {word}")
                            + f". My facts form {len(tops)} groups I can't compare with each other, so I can't "
                              f"say outright.")
        names = [self._say(t[0]) for t in tops]
        return (f"I can't say for sure: my facts form {len(tops)} separate groups, and the top of each is "
                f"{', '.join(names[:-1])} or {names[-1]}. Tell me how the groups relate and I'll know.")

    def _ask_rank(self, p):
        axis, sign = p["axis"], p["sign"]
        space = L.space_of(axis)
        if space not in self.maps or axis not in self.maps[space].rel:
            return f"You haven't told me anything about {axis} yet."
        self._settle(space)
        m = self.maps[space]
        word = self.words.get(axis, L.positive_word(axis)) if space != "place" else L.SAY_AXIS[axis][0]
        lines = []
        for g in self._groups(space):
            order = sorted(g, key=lambda n: -sign * float(m.pos[m.index[n]] @ m.rel[axis]))
            parts = [self._say(order[0])]
            for prev, n in zip(order, order[1:]):
                gap = abs(m.steps(n, axis, prev))
                parts.append(("ÃƒÂ¢Ã¢â‚¬Â°Ã‹â€  " if gap < 0.5 else "> ") + self._say(n))
            lines.append(" ".join(parts))
        if space == "place":
            pos, neg = L.SAY_AXIS[axis]
            head = f"From {pos}most to {neg}most" if axis in ("north", "east") else f"From most {pos} to most {neg}"
            if sign < 0:
                head = f"From {neg}most to {pos}most" if axis in ("north", "east") else f"From most {neg} to most {pos}"
        else:
            head = f"By {self._dim(axis)}, {word} first" if sign == 1 else f"By {self._dim(axis)}, {word} last"
        if len(lines) == 1:
            return f"{head}: {lines[0]}"
        return head + f" - {len(lines)} separate groups I can't compare with each other:\n" + "\n".join(
            "  ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢ " + l for l in lines)

    def _ask_between(self, p):
        (ka, _), (kb, _) = p["a"], p["b"]
        miss = self._unknown(ka, kb)
        if miss:
            return miss
        found = []
        for space, m in self.maps.items():
            if not self._connected(space, ka, kb):
                continue
            self._settle(space)
            pa, pb = m.pos[m.index[ka]], m.pos[m.index[kb]]
            d = pb - pa
            L2 = float(d @ d)
            if L2 < 0.25:
                continue
            for n in m.names:
                if n in (ka, kb) or not self._connected(space, ka, n):
                    continue
                v = m.pos[m.index[n]] - pa
                t = float(v @ d) / L2
                off = float((v - t * d).norm())
                length = L2 ** 0.5
                if 0.5 / length < t < 1 - 0.5 / length and off < 0.5:
                    found.append((t, space, n))
        if not found:
            if not any(self._connected(s, ka, kb) for s in self.maps):
                return self._not_linked(ka, kb, "at all")
            return f"Nothing I know of lies between {self._say(ka)} and {self._say(kb)}."
        by = {}
        for _, space, n in sorted(found):
            by.setdefault(space, []).append(self._say(n))
        parts = [f"{', '.join(v)}" + ("" if s == "place" else f" (by {s})") for s, v in by.items()]
        return "Between them: " + "; ".join(parts) + "."

    def _ask_howmuch(self, p):
        (ka, _), (kb, _) = p["a"], p["b"]
        axis, sign, word = p["axis"], p["sign"], p["word"]
        miss = self._unknown(ka, kb)
        if miss:
            return miss
        space = L.space_of(axis)
        why = self._link_check(space, ka, kb, self._dim(axis))
        if why:
            return why
        self._settle(space)
        s = self.maps[space].steps(kb, axis, ka) * sign
        n = round(abs(s))
        unit = "step" if n == 1 else "steps"
        if s < -0.5:
            return f"{self._say(ka)} isn't {word} - {self._say(kb)} is, by about {n} {unit}."
        if abs(s) < 0.5:
            return f"About the same ({s:+.1f} steps)."
        return (f"About {n} {unit} (exactly {s:.2f}). A step is one \"{word} than\" fact - you told me orderings, "
                f"not amounts, so I count links in the chain.")

    def _describe_offset(self, off):
        parts = []
        for axis in L.SPATIAL_AXES:
            v = off.get(axis, 0.0)
            if abs(v) >= 0.5:
                n = round(abs(v))
                pos, neg = L.SAY_AXIS[axis]
                w = pos if v > 0 else neg
                parts.append(f"{n} {w}" if axis in ("north", "east") else f"{n} step{'s' * (n != 1)} {w}")
        return " and ".join(parts) if parts else "in the same spot"

    def _ask_where(self, p):
        (ka, _), (kb, _) = p["a"], p["b"]
        miss = self._unknown(ka, kb)
        if miss:
            return miss
        why = self._link_check("place", ka, kb, "where things are")
        if why:
            return why
        self._settle("place")
        off = self._offset("place", kb, ka)
        return f"{self._say(ka)} is {self._describe_offset(off)} of {self._say(kb)}.".replace(
            "in the same spot of", "in the same spot as")

    def _ask_direction(self, p):
        (ka, _), (kb, _) = p["a"], p["b"]
        miss = self._unknown(ka, kb)
        if miss:
            return miss
        why = self._link_check("place", ka, kb, "where things are")
        if why:
            return why
        self._settle("place")
        off = self._offset("place", kb, ka)
        actual = self._describe_offset(off)
        if p["exact"]:
            want = dict(p["moves"])
            ok = all(abs(off.get(a, 0.0) - want.get(a, 0)) < 0.5 for a in set(off) | set(want))
        else:
            ok = all(off.get(a, 0.0) * c > 0.5 for a, c in p["moves"])
        verdict = "Yes" if ok else "No"
        return f"{verdict} - {self._say(ka)} is {actual} of {self._say(kb)}.".replace(
            "in the same spot of", "in the same spot as")

    def _ask_at(self, p):
        (ka, _) = p["a"]
        miss = self._unknown(ka)
        if miss:
            return miss
        if "place" not in self.maps or not self.maps["place"].knows(ka):
            return f"I don't know where {self._say(ka)} is."
        self._settle("place")
        m = self.maps["place"]
        for r, _ in p["moves"]:
            if r not in m.rel:
                return f"You haven't told me anything {L.SAY_AXIS[r][0]} or {L.SAY_AXIS[r][1]} of anything yet."
        target = m.pos[m.index[ka]] + m._disp(tuple(p["moves"]))
        near = [(n, d) for n, d in m.nearest(target, top=len(m.names)) if self._connected("place", ka, n)]
        if not near:
            return "Nothing I know of."
        n, d = near[0]
        if n == ka and d < 0.5:
            return f"That brings you back to {self._say(ka)}."
        if d < 0.5:
            return f"{self._say(n)}."
        if d < 1.0:
            return f"Probably {self._say(n)} - it's the closest thing to that spot ({d:.1f} steps off)."
        return f"Nothing I know of is there. The closest is {self._say(n)}, {d:.1f} steps away."

    def _ask_about(self, p):
        (ka, sa) = p["a"]
        miss = self._unknown(ka)
        if miss:
            return miss
        lines = []
        for space, m in self.maps.items():
            if not m.knows(ka):
                continue
            self._settle(space)
            group = [n for n in m.names if self._connected(space, ka, n)]
            if space == "place":
                near = [(n, d) for n, d in m.nearest(m.pos[m.index[ka]], top=4) if n != ka and n in group][:3]
                if near:
                    rel = "; ".join(f"{self._describe_offset(self._offset('place', n, ka))} of {self._say(n)}"
                                    for n, _ in near)
                    lines.append(f"Place: {rel}.")
            else:
                axis = next(iter(m.rel))
                order = sorted(group, key=lambda n: -float(m.pos[m.index[n]] @ m.rel[axis]))
                word = self.words.get(axis, L.positive_word(axis))
                lines.append(f"{self._dim(axis).capitalize()}: {ordinal(order.index(ka) + 1)} of {len(order)}, "
                             f"counting from the {word} end.")
        return f"{self._say(ka)}:\n" + "\n".join(lines) if lines else f"I know {self._say(ka)} but nothing more."

    # ------------------------------------------------------------------ commands
    def _cmd_sleep(self):
        if not self.maps:
            return "Nothing to consolidate yet."
        moved = 0.0
        for space, m in self.maps.items():
            before = m.pos.clone()
            m.sleep(self.config["map_config"]["sleep"], tolerance=self.config["map_config"]["tol"])
            # recentering shifts everything rigidly; measure how much the shape changed
            delta = (m.pos - before)
            delta = delta - delta.mean(0) if len(delta) else delta
            moved = max(moved, float(delta.norm(dim=1).max()) if len(delta) else 0.0)
        self.dirty.clear()
        n = sum(len(m.episodes) for m in self.maps.values())
        return (f"Slept: replayed all {n} remembered facts together and let the map settle. "
                f"The biggest change to any position was {moved:.2f} steps.")

    def _cmd_reset(self):
        self.reset()
        return "Forgotten. My maps are empty again."

    def _cmd_summary(self):
        if not self.maps:
            return "Nothing yet - my maps are empty. Tell me something!"
        lines = []
        for space, m in self.maps.items():
            groups = self._groups(space)
            label = "Places" if space == "place" else self._dim(space).capitalize()
            lines.append(f"ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¢ {label}: {int(m.placed.sum())} things from {len(m.episodes)} facts"
                         + (f", in {len(groups)} unlinked groups" if len(groups) > 1 else ""))
        total = sum(m.footprint() for m in self.maps.values())
        return ("Here's what I've learned (all of it from you):\n" + "\n".join(lines) +
                f"\nMy whole memory is {total} numbers - a position per thing and a direction per relation.")

    def _cmd_help(self):
        return HELP

    # ------------------------------------------------------------------ view
    def view(self):
        """2-D picture of every map for the interface: places by east/north, each dimension as a ladder."""
        spaces = []
        for space, m in self.maps.items():
            live = [n for n in m.names if m.knows(n)]
            if not live:
                continue
            idx = torch.tensor([m.index[n] for n in live])
            P = m.pos[idx]
            if space == "place":
                ax = [a for a in ("east", "north") if a in m.rel]
                xs = (P @ m.rel["east"]) if "east" in m.rel else torch.zeros(len(live))
                ys = (P @ m.rel["north"]) if "north" in m.rel else torch.zeros(len(live))
                if "north" not in m.rel and "up" in m.rel:
                    ys = P @ m.rel["up"]
                if "east" not in m.rel and "front" in m.rel:
                    xs = P @ m.rel["front"]
                kind = "place"
            else:
                axis = next(iter(m.rel))
                ys = P @ m.rel[axis]
                # spread items at the same level sideways so they don't overlap
                xs = torch.zeros(len(live))
                kind = "line"
                ax = [axis]
            points = [{"key": n, "name": self._say(n), "x": float(x), "y": float(y), "group": int(m.island[m.index[n]])}
                      for n, x, y in zip(live, xs, ys)]
            edges = [{"a": m.names[a], "b": m.names[b]} for a, _, b in m.episodes]
            label = "Places" if space == "place" else self._dim(space).capitalize()
            word = self.words.get(space, L.positive_word(space)) if kind == "line" else None
            spaces.append({"space": space, "label": label, "kind": kind, "axes": ax, "word": word,
                           "points": points, "edges": edges})
        return spaces


def article(shown):
    """'The Cafe' -> 'the Cafe', so names read naturally mid-sentence (replies are capitalised at the start)."""
    return re.sub(r"^(The|A|An)\s", lambda m: m.group(1).lower() + " ", shown)


def ordinal(n):
    return f"{n}{'th' if 10 <= n % 100 <= 20 else {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')}"


def describe(p):
    """The parse, shown under each reply so it is always clear what the model was actually given."""
    k = p["kind"]
    if k == "fact":
        mv = " + ".join(f"{c:+d} {a}" for a, c in p["moves"])
        return f"learn: {p['a'][0]} = {p['b'][0]} {mv}"
    if k == "ask":
        bits = [p["q"]] + [f"{f}={p[f][0]}" for f in ("a", "b") if p.get(f)]
        if "axis" in p:
            bits.append(f"axis={p['axis']}{'+' if p['sign'] > 0 else '-'}")
        if "moves" in p:
            bits.append("moves=" + ",".join(f"{c:+d}{a}" for a, c in p["moves"]))
        return "ask: " + " ".join(bits)
    if k == "command":
        return "command: " + p["do"]
    return "not understood"

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/__init__.py
"""A conversational brain: lexical cortex, prefrontal working memory, hippocampal map and episodic store, a
basal-ganglia gate that learns which of them answers, and sleep. No pretrained weights, no gradients."""

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/codes.py
"""Lexical cortex: the only way anything enters this system.

A word is not a string here and not a row in a pretrained table - it is a sparse pattern of active units, produced
by fixed random wiring plus k-winners-take-all, exactly as in the mushroom body and the dentate gyrus. Two
properties follow, and everything downstream depends on both:

  overlap means similarity   words sharing letter trigrams drive overlapping inputs, so "dog" and "dogs" land on
                             codes that share units. Nothing was trained to make that true; it falls out of the
                             wiring, which is what morphology-by-similarity looks like in tissue.
  different means separate   unrelated words barely overlap, so binding one into memory does not disturb another.

The vocabulary is unbounded: a word is hashed into a fixed number of input slots, so a word never seen before has
a code the moment it is heard. There is no table to grow and no out-of-vocabulary case.
"""
import zlib

import torch

from brain.column import Column

SLOTS = 4096       # input fibres the lexicon projects onto
N_UNITS = 8192     # lexical cortex units
K_ACTIVE = 64      # how many fire: 0.8%, the sparsity cortex runs at


def _slot(text, salt=0):
    """A stable hash - Python's own is salted per process, and a brain's wiring does not change overnight."""
    return zlib.crc32(text.encode("utf-8"), salt) % SLOTS


def trigrams(word):
    w = f"<{word}>"
    return [w[i:i + 3] for i in range(len(w) - 2)] or [w]


class Lexicon:
    """Words and phrases -> sparse codes. One cortical column, wired to the input fibres; never trained.

    Because summation inside a branch is supralinear, a term whose parts arrive together on the same branch
    drives its units harder than the same evidence spread thinly - so a phrase is not the average of its words.
    """

    def __init__(self, n=N_UNITS, k=K_ACTIVE, seed=0, homeostasis=0.01, branches=4):
        self.column = Column("lexical", SLOTS, n=n, k=k, branches=branches, seed=seed, homeostasis=homeostasis)
        self.expand = lambda v: self.column.encode(v)
        self.n, self.k = n, k
        self._cache = {}

    def drive(self, term):
        """The input pattern a term causes: its identity, plus its letter trigrams so morphology overlaps."""
        v = torch.zeros(SLOTS)
        for word in term.lower().split():
            v[_slot(word, 1)] += 1.5                 # the word itself, weighted a little above its parts
            for g in trigrams(word):
                v[_slot(g, 2)] += 1.0                # its letter trigrams: what makes "dog" and "dogs" overlap
        return v

    def code(self, term):
        """A term -> its sparse code (n,), k units active."""
        if term not in self._cache:
            self._cache[term] = self.column.encode(self.drive(term))
        return self._cache[term]

    def bag(self, terms, weights=None):
        """Several terms at once -> one sparse code. Used for an utterance: the code of what was just said.

        The drives are summed before the winner-take-all, not the codes afterwards, so the result is a genuine
        population response to the whole phrase rather than a union of word codes - overlapping evidence adds up
        and the k winners are whatever the phrase as a whole drives hardest.
        """
        if not terms:
            return torch.zeros(self.n)
        v = torch.zeros(SLOTS)
        for t, w in zip(terms, weights or [1.0] * len(terms)):
            v += w * self.drive(t)
        return self.column.encode(v)

    def nearest(self, code, candidates, top=3):
        """Population readout: which known terms this pattern overlaps most. How a code becomes a word again."""
        if not candidates:
            return []
        scored = [(t, float(self.code(t) @ code) / self.k) for t in candidates]
        scored.sort(key=lambda x: -x[1])
        return scored[:top]

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/column.py
"""The canonical microcircuit, which every area is made of.

Cortex is not a collection of bespoke machines. The same six-layer circuit is repeated across areas that do
utterly different jobs - vision, touch, planning - and what differs between them is what they are wired to, how
big they are and how they are tuned, not the circuit. Subcortical structures repeat their own motif the same
way. That uniformity is the strongest architectural fact about the brain, and this file is the project taking it
literally: one `Column`, used for every store.

  input layer (L4)      dendritic expansion - coincidence detection on branches, divisive gain control, k-WTA
  local layer (L2/3)    lateral inhibition: only the k best-driven units carry the pattern onward
  output layer (L5)     a local associative readout - Hebbian counts from the active units to whatever the
                        column is being asked to produce, and nothing else

A column that stores patterns (episodic), a column that names them (lexical), a column that selects an action
(the gate) are the same object with different sizes and different targets. If the microcircuit is wrong, it is
wrong everywhere at once, which is the point: it can be tested in one place.
"""
import torch

from brain.dendrite import Dendrites


class Column:
    def __init__(self, name, d_in, n=8192, k=64, branches=4, n_out=0, seed=0, **kw):
        self.name, self.d_in, self.n, self.k = name, d_in, n, k
        self.dendrites = Dendrites(d_in, n=n, k=k, branches=branches, seed=seed, **kw)
        self.out = torch.zeros(n, n_out) if n_out else None     # L5: local Hebbian associations
        self.fired = 0

    def encode(self, x, learn=True):
        """Drive -> sparse code. The only way anything enters a column."""
        single = x.dim() == 1
        code = self.dendrites(x[None] if single else x, learn=learn)
        self.fired += code.shape[0]
        return code[0] if single else code

    def associate(self, code, target, strength=1.0):
        """Hebbian: the units that fired grow their synapses onto what was present at the same moment."""
        if self.out is None:
            raise ValueError(f"column {self.name} has no output layer")
        self.out[code > 0, target] += strength

    def read(self, code):
        return code @ self.out if self.out is not None else None

    def spec(self):
        s = {"column": self.name, "inputs": self.d_in, **self.dendrites.spec(), "fired": self.fired}
        if self.out is not None:
            s["output_units"] = int(self.out.shape[1])
            s["output_synapses"] = int((self.out != 0).sum())
        return s

    def synapses(self):
        return int(self.dendrites.idx.numel()) + (int((self.out != 0).sum()) if self.out is not None else 0)

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/dendrite.py
"""Dendritic summation: what a neuron does to its inputs before it decides to fire.

The expansion this project has used until now sums every synapse linearly and takes the k largest. A real neuron
does neither of those things, and the two differences change what the code means:

  non-linear summation   synapses landing close together on the same dendritic branch interact. Two inputs on
                         one branch produce more than twice one input (NMDA-dependent supralinearity), and the
                         branch then saturates. Two inputs on *different* branches just add. So a unit is not a
                         weighted sum - it is a small layer of coincidence detectors, one per branch, and what
                         it is tuned to is a *combination* of inputs arriving together on the same branch.
  suppression            inhibition here is not a bigger negative number. Perisomatic (PV-like) inhibition
                         divides the whole drive by how much the population as a whole is active - gain control,
                         so a loud input does not simply recruit more units. Dendrite-targeting (SST-like)
                         inhibition subtracts from *specific* branches that have been active recently, which is
                         input-specific adaptation rather than global quieting.

Both are local: a branch sees only its own synapses, the divisive term sees only the population mean, and the
adaptation term sees only each unit's own recent activity. Nothing is trained.
"""
import math

import torch


class Dendrites:
    """d_in -> n units with `branches` dendritic branches each; exactly k fire."""

    def __init__(self, d_in, n=8192, k=64, fan_in=12, branches=4, seed=0, theta=1.0, divisive=0.5,
                 adapt=0.0, homeostasis=0.0):
        g = torch.Generator().manual_seed(int(seed))
        self.d_in, self.n, self.k, self.branches = d_in, n, k, branches
        # every input must reach enough units, or a lone active input cannot fill the k winners and the rest are
        # ties at zero - the same filler units for every input, which makes codes collide
        fan_in = max(fan_in, branches, math.ceil(2 * k * d_in / n))
        fan_in += (-fan_in) % branches                      # divide evenly between branches
        wiring = torch.arange(n * fan_in) % d_in
        self.idx = wiring[torch.randperm(n * fan_in, generator=g)].reshape(n, branches, fan_in // branches)
        self.w = 0.5 + torch.rand(n, branches, fan_in // branches, generator=g)
        self.theta, self.divisive, self.adapt = theta, divisive, adapt
        self.bias = torch.zeros(n)
        self.fatigue = torch.zeros(n, branches)             # SST-like, branch-specific, from recent activity
        self.rate = torch.full((n,), k / n)
        self.homeostasis = homeostasis

    @torch.no_grad()
    def __call__(self, x, learn=True):
        """x (B, d_in) -> binary code (B, n), exactly k ones."""
        s = (x[:, self.idx] * self.w).sum(-1)               # (B, n, branches): drive per branch
        s = (s - self.fatigue).clamp_min(0.0)               # branch-specific suppression
        # supralinear then saturating: s^2 / (theta + s). Small inputs are discounted, coincident ones on the
        # same branch multiply, and a branch cannot shout indefinitely.
        branch = s * s / (self.theta + s)
        drive = branch.sum(-1) - self.bias                  # branches add linearly at the soma
        if self.divisive:                                   # PV-like gain control: divide by how loud the field is
            drive = drive / (1.0 + self.divisive * drive.mean(dim=1, keepdim=True))
        code = torch.zeros(x.shape[0], self.n)
        code.scatter_(1, drive.topk(self.k, dim=1).indices, 1.0)
        if learn and self.adapt:                            # branches that just fired are harder to drive next
            active = (code[..., None] * (branch > 0).float())
            self.fatigue.mul_(1 - self.adapt).add_(self.adapt * active.mean(0) * s.mean(0))
        if learn and self.homeostasis:
            self.rate.mul_(0.99).add_(0.01 * code.mean(0))
            self.bias.add_(self.homeostasis * (self.rate - self.k / self.n))
        return code

    def spec(self):
        return {"units": self.n, "active": self.k, "branches": self.branches,
                "synapses_per_branch": int(self.idx.shape[2]), "divisive": self.divisive, "adapt": self.adapt}

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/episodic.py
"""Hippocampal episodic store: every utterance, kept once and recalled from a partial cue.

The map holds what is true. This holds what happened - who said what, in which words, at which turn. They are
different memories in the brain and they are different here, which is the whole point of the redesign: one store
was being asked to give content words broad neighbourhoods and frame words sharp ones, and no single width can.

  pattern separation   an utterance is expanded into a sparse code by fixed wiring, so two similar sentences get
                       codes that overlap only as far as they genuinely share words (dentate gyrus).
  pattern completion   recall settles: the best-matching memories vote, the vote is re-sharpened to exactly k
                       active units, and the match is run again. A cue that is partly wrong still lands on the
                       right memory once its neighbours agree (CA3 recurrence).
  no overwriting       storing is appending a code. Nothing is averaged, so one odd sentence cannot smear the
                       memory of another - the failure mode that dense stores have and this does not.
"""
import torch

from brain.column import Column


class Episodes:
    def __init__(self, lexicon, n=16384, k=96, seed=5, branches=6):
        self.lex = lexicon
        # the same column as everywhere else, wired to the lexical code and tuned for separation rather than
        # generalisation: more units, more branches, so utterances that differ in a word land far apart
        self.column = Column("episodic", lexicon.n, n=n, k=k, branches=branches, seed=seed)
        self.sep = lambda v: self.column.encode(v)
        self.n, self.k = n, k
        self.codes = torch.zeros(0, n)
        self.records = []

    def _code(self, text, turn=0):
        return self.column.encode(self.lex.bag(text.lower().split()))

    def store(self, text, turn, kind, payload=None):
        """One utterance, stored once."""
        self.codes = torch.cat([self.codes, self._code(text, turn)[None]])
        self.records.append({"turn": turn, "text": text, "kind": kind, "payload": payload})
        return len(self.records) - 1

    def recall(self, cue, rounds=2, top=5):
        """-> (record, confidence). The cue does not have to be anything that was said."""
        if not len(self.codes):
            return None, 0.0
        c = self._code(cue, 0)
        for _ in range(rounds):
            score = self.codes @ c
            best = score.topk(min(top, len(self.codes))).indices
            if float(score[best[0]]) <= 0:
                return None, 0.0
            w = score[best] / score[best].sum().clamp_min(1e-9)
            vote = (self.codes[best] * w[:, None]).sum(0)
            c = torch.zeros_like(c)
            c[vote.topk(self.k).indices] = 1.0     # back to exactly k units: a memory, not a blur
        score = self.codes @ c
        i = int(score.argmax())
        return self.records[i], float(score[i] / self.k)

    def about(self, term, limit=4):
        """Every episode whose code overlaps this term's - what was said about something."""
        if not len(self.codes):
            return []
        c = self.column.encode(self.lex.code(term))
        score = self.codes @ c
        order = score.argsort(descending=True)[:limit]
        return [(self.records[int(i)], float(score[i] / self.k)) for i in order if score[i] > 0]

    def replay(self, count=8, rng=None):
        """Sequences for consolidation: recent episodes come up most, as in sharp-wave ripples."""
        if not self.records:
            return []
        n = min(count, len(self.records))
        weights = torch.linspace(0.3, 1.0, len(self.records))
        idx = torch.multinomial(weights, n, replacement=False)
        return [self.records[int(i)] for i in sorted(idx.tolist())]

    def synapses(self):
        return int((self.codes > 0).sum()) + self.column.synapses()

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/gate.py
"""Basal-ganglia gate: which system is allowed to answer.

The old router was a hand-ordered cascade of `if` statements - try the map, then the web, then give up - which is
a programmer's decision frozen into the code, and it never got better. Selecting between competing systems is
what basal ganglia circuitry is for, and it is *learned*: cortical loops propose, the striatum releases one, and
what happened next changes which one is released next time.

Here the proposers are the stores - working memory, the episodic store, the cognitive map - and the selector is
an expert pool over a sparse code of the situation:

  the situation   not just what was asked, but what each store currently has: does the map know both things, are
                  they connected, did an episode match strongly, is there an unresolved pronoun. A gate that saw
                  only the question could never learn that the same question needs a different system depending
                  on what is known.
  selection       the experts wired to the active units vote, the strongest wins (lateral inhibition).
  learning        an action that produces grounded evidence reinforces the expert that chose it. One that comes
                  back empty raises the match threshold above that expert and the search continues - ARTMAP match
                  tracking - so the situation gets its own expert instead of overwriting the old one.
  neurogenesis    a situation unlike anything seen recruits a brand-new expert, wired to exactly the units that
                  are active. The gate grows with use; it is never trained.

Before anything has been learned the gate has a reflex: a fixed prior from the parser's intent. Every reflex that
works is immediately learned from, so the pool takes over from the reflexes within a few turns.
"""
import torch

from neuromoe import ExpertPool

ACTIONS = ("teach", "infer", "recall", "describe", "clarify", "social")

# what the parser's intent asks for, before any experience: the innate part, which experience overrules
REFLEX = {"fact": "teach", "compare": "infer", "which": "infer", "extreme": "infer", "rank": "infer",
          "between": "infer", "howmuch": "infer", "at": "infer", "where": "infer", "direction": "infer",
          "about": "describe", "recall": "recall", "command": "social", "social": "social", "unknown": "clarify"}


class Gate:
    def __init__(self, lexicon, vigilance=0.55, trust=1.0, seed=7):
        self.lex = lexicon
        self.pool = ExpertPool(lexicon.n, len(ACTIONS), vigilance=vigilance, k_route=3, beta=0.25,
                               labels="exact", trust=trust)
        self.chosen = {a: 0 for a in ACTIONS}

    def situation(self, intent, signals):
        """A sparse code of the whole situation: what was asked, and what each store has to offer.

        Only signals that have *arrived* are in here. A store whose pathway is slower than the deadline
        contributes nothing to the choice, because its answer is still in transit.
        """
        terms = ["\x00intent:" + intent] + ["\x00sig:" + s for s, on in sorted(signals.items()) if on]
        return self.lex.bag(terms)

    def choose(self, intent, signals, allowed):
        """-> (action, confidence, how). `how` is 'learned' when an expert claimed it, 'reflex' otherwise.

        `allowed` is what this turn could physically be answered by - you cannot teach a question or infer from a
        greeting. The gate never votes on impossible actions, in the same way basal ganglia loops select among
        motor programs that are actually available rather than among all conceivable ones. Everything genuinely
        open - describe it, recall it, or work it out - is left to the experts and to what happens next.
        """
        code = self.situation(intent, signals)
        allowed = [a for a in allowed if a in ACTIONS] or ["clarify"]
        reflex = REFLEX.get(intent, "clarify")
        if reflex not in allowed:
            reflex = allowed[0]
        if not len(self.pool):
            return reflex, 0.0, "reflex", code
        p, match = self.pool.predict(code, return_match=True)
        mask = torch.zeros(len(ACTIONS))
        for a in allowed:
            mask[ACTIONS.index(a)] = 1.0
        p = p * mask
        confidence = float(p.max())
        action = ACTIONS[int(p.argmax())]
        if match < self.pool.vigilance or confidence < 0.4:
            return reflex, confidence, "reflex", code
        return action, confidence, "learned", code

    def reinforce(self, code, action):
        """That action found something. The expert that chose it grows toward this situation."""
        self.pool.learn(code, ACTIONS.index(action))
        self.chosen[action] += 1

    def correct(self, code, failed, worked):
        """That action came back empty and another one did not. Match tracking gives this situation its own
        expert rather than blurring the one that was right elsewhere."""
        if worked is None:
            return
        self.pool.learn(code, ACTIONS.index(worked))
        self.chosen[worked] += 1

    def stats(self):
        return {"experts": len(self.pool), "born": self.pool.born, "died": self.pool.died,
                "synapses": self.pool.synapses(), "chosen": dict(self.chosen)}

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/semantic.py
"""Semantic cortex: the slow store, and background knowledge if any has been built.

Complementary learning systems, which is the oldest well-supported division in memory research: the hippocampus
takes a fact in one exposure and keeps it apart from everything else; cortex learns slowly, overlapping, and ends
up with structure rather than instances. This is the cortical half.

Two things live here:

  what has been consolidated   facts the conversation established, replayed out of the episodic store during
                               sleep and written into a map that is allowed to blur them into structure.
  what was built beforehand    the ConceptNet/prose map, if it exists on this machine. It is background, not the
                               centre: the gate may consult it, and nothing forces an answer through it.

The prebuilt map is optional on purpose. Everything the system does about *this conversation* works on a machine
that has never seen a corpus.
"""
import os

import torch

PREBUILT = os.environ.get("CONCEPTS", "E:/corpora/concepts.pt")


class Semantic:
    def __init__(self, path=PREBUILT, top_terms=120000):
        self.pos = self.names = None
        self.rel = {}
        self.index = {}
        self.vocab = set()
        if path and os.path.exists(path):
            d = torch.load(path, weights_only=False)
            self.pos, self.names, self.rel = d["pos"], d["names"], d["rel"]
            self.index = {n: i for i, n in enumerate(self.names)}
            self.vocab = {n for n in self.names[:top_terms] if "_" not in n}

    @property
    def loaded(self):
        return self.pos is not None

    def knows(self, term):
        return term.lower() in self.index

    def neighbours(self, term, k=6):
        """Place-cell readout over concepts: what sits nearest, which is what this store is good at."""
        t = term.lower()
        if not self.loaded or t not in self.index:
            return []
        dist = (self.pos - self.pos[self.index[t]]).norm(dim=1)
        out = []
        for i in dist.argsort()[1:k * 8]:
            n = self.names[int(i)]
            if "_" in n or n.startswith(t[:max(3, len(t) - 1)]):
                continue      # morphology of the same word is a fact about English, not about the thing
            out.append((n, float(dist[i])))
            if len(out) >= k:
                break
        return out

    def along(self, term, relation, k=4):
        """One step of a relation from a term - inference by displacement, the map's own move."""
        t = term.lower()
        if not self.loaded or t not in self.index or relation not in self.rel:
            return []
        target = self.pos[self.index[t]] + self.rel[relation]
        dist = (self.pos - target).norm(dim=1)
        out = []
        for i in dist.argsort()[:k * 8]:
            n = self.names[int(i)]
            if "_" in n or n == t:
                continue
            out.append((n, float(dist[i])))
            if len(out) >= k:
                break
        return out

    def describe(self, term, relations=("IsA", "PartOf", "UsedFor", "AtLocation", "CapableOf", "HasProperty")):
        t = term.lower()
        if not self.knows(t):
            return None
        near = self.neighbours(t)
        out = {"term": t, "near": near, "relations": {}}
        here = {n for n, _ in near}
        for r in relations:
            hits = [h for h in self.along(t, r) if h[0] not in here]
            if hits:
                out["relations"][r] = hits[:3]
        return out

    def size(self):
        return 0 if not self.loaded else len(self.names)

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/speech.py
"""Putting an answer into words.

Deliberately thin. Every sentence here is composed from a structure some store produced - there is no template
bank standing in for knowing something, and nothing is said that no store supports. When the systems have
nothing, the honest sentence is the whole reply.
"""


def listify(items):
    items = [str(i) for i in items]
    if len(items) <= 1:
        return "".join(items)
    return ", ".join(items[:-1]) + " and " + items[-1]


def greeting(known, facts):
    if not facts:
        return ("Hello. I don't know anything yet - I have no pretrained knowledge at all. Tell me how things "
                "relate and I'll work out the rest.")
    return f"Hello again. I'm holding {facts} fact(s) from what you've told me."


def acknowledge(topic, recalled):
    """Someone said something about themselves or the world that is not a relation. Say what is actually held."""
    if recalled:
        return f"I hear you. You mentioned {topic} before - {recalled}."
    if topic:
        return f"I hear you. I don't have anything about {topic} beyond what you've just said."
    return "I hear you. I can't do much with that yet - I hold relations between things."


def clarify(reason, subject=None):
    if reason == "pronoun":
        return "Who or what do you mean? I've lost track of who that refers to."
    if reason == "unparsed":
        return ("I didn't follow that. I understand facts like \"Alice is taller than Bob\" or \"the cafe is "
                "north of the park\", and questions about them.")
    if reason == "unknown-term" and subject:
        return f"I don't know anything about {subject} yet. Tell me how it relates to something else."
    return "I'm not sure what you're asking."


def recalled(record, confidence, turn_gap):
    when = "just now" if turn_gap <= 2 else f"{turn_gap} turns ago"
    hedge = "" if confidence > 0.55 else " (though the memory is faint)"
    return f"You said {when}: \"{record['text']}\"{hedge}."


def about_semantic(term, detail):
    bits = []
    for rel, hits in list(detail["relations"].items())[:3]:
        bits.append(f"{rel}: " + listify([h[0] for h in hits[:2]]))
    line = f"From background knowledge, {term} - " + "; ".join(bits) if bits else ""
    if detail["near"]:
        near = listify([n for n, _ in detail["near"][:3]])
        line = (line + ". " if line else f"I only know {term} by its company: ") + f"it sits near {near}."
    return line


def sleep_report(facts, moved, experts):
    return (f"Slept. I replayed {facts} fact(s) together, the biggest change to any position was {moved:.2f} "
            f"steps, and my gate is holding {experts} expert(s).")

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/tracts.py
"""Conduction delay, which is where sequence comes from.

Nothing in a brain happens "next" because a program said so. A spike leaves a cell and takes time to arrive -
about a millisecond per millimetre in unmyelinated cortex - so the order in which regions hear about an event is
set by how far apart they are. Fast, short pathways deliver a rough answer first; long ones deliver a better one
later. That is why there are reflexes at all, and why deciding early and deciding well are different things.

Until now the order of this project's turn loop was the order the Python happened to be written in. Here it is
distance:

    lexical --1--> working memory --1--> gate
        \\--2--> episodic --1-->/
         \\--------4-------> map ---1--->/

The gate decides at a deadline. Evidence that has not arrived by then cannot affect the choice - it is not
ignored, it is *not there yet*. Shorten the deadline and the same question is answered by whichever store is
nearest; lengthen it and the slow, careful store gets its say. Both are the same circuit run under different
urgency, which is what the two-pathway story in the brain actually is.
"""
import math

# where each region sits, in conduction units. Only the distances matter.
LAYOUT = {"lexical": 0.0, "working": 1.0, "gate": 2.0, "episodic": 2.5, "semantic": 3.0, "map": 4.5}
DEADLINE = 6            # ticks the gate will wait before it has to choose


class Tracts:
    def __init__(self, layout=None, speed=1.0, deadline=DEADLINE):
        self.layout = dict(layout or LAYOUT)
        self.speed = speed
        self.deadline = deadline

    def delay(self, src, dst):
        """Ticks for a signal to travel, never less than one: nothing is instantaneous."""
        d = abs(self.layout[dst] - self.layout[src]) / self.speed
        return max(1, int(math.ceil(d)))

    def arrival(self, src, dst, via=None):
        """When a signal sent at tick 0 from `src` reaches `dst`, optionally relayed through other regions."""
        t, here = 0, src
        for hop in list(via or []) + [dst]:
            t += self.delay(here, hop)
            here = hop
        return t

    def in_time(self, src, dst, via=None, deadline=None):
        return self.arrival(src, dst, via) <= (self.deadline if deadline is None else deadline)

    def timeline(self, src="lexical"):
        """What arrives when, for the panel and for the record of each turn."""
        return sorted(((self.arrival(src, r), r) for r in self.layout if r != src))

    def spec(self):
        return {"layout": self.layout, "speed": self.speed, "deadline": self.deadline,
                "arrivals": {r: self.arrival("lexical", r) for r in self.layout if r != "lexical"}}

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/working.py
"""Prefrontal working memory: what is being talked about right now.

This is the piece whose absence made the old portal unable to hold a conversation. It is not a dictionary of
"context variables" - it is the same one-shot associative memory the benchmarks showed recalls a thousand pairs
with no training (M += beta (v - M k) k^T / |k|), used the way prefrontal cortex is thought to use persistent
activity: a few things held active, decaying unless refreshed, addressable by content.

  slots        topic, who was mentioned, the last relation, the last answer. Each slot is a fixed random key
               code; writing a binding is one local update, reading it is one matrix-vector product.
  decay        bindings fade unless rewritten, so "she" means whoever was last talked about rather than whoever
               was talked about an hour ago. This is a leak, not a queue - nothing is deleted on a schedule.
  time cells   every turn gets its own sparse code, so an utterance can be bound to *when* it was said and the
               conversation can be replayed in order.

Reading a slot gives a pattern, not a string. Turning it back into a word is a population readout over the terms
the conversation has actually used - so the memory can be wrong, and says how strongly it is bound.
"""
import torch

from neuromoe import TimeCells, WorkingMemory

SLOTS = ("topic", "subject", "object", "relation", "answer", "question", "unresolved")


class Prefrontal:
    def __init__(self, lexicon, decay=0.94, max_turns=4096, seed=3):
        self.lex = lexicon
        self.n = lexicon.n
        # bindings of slot -> content, and of turn -> what was said then
        self.slots = WorkingMemory(self.n, self.n, beta=1.0, decay=decay)
        self.episodes = WorkingMemory(self.n, self.n, beta=1.0, decay=1.0)
        self.time = TimeCells(n=self.n, k=lexicon.k, seed=seed, max_steps=max_turns)
        self.slot_key = {s: lexicon.code("\x00slot:" + s) for s in SLOTS}
        self.turn = 0
        self.mentioned = []      # terms this conversation has used, most recent first: the readout's candidates
        self.history = []        # [(turn, text, kind)] - the transcript, for grounding what it claims to recall

    # ------------------------------------------------------------------ holding
    def hold(self, slot, term):
        """Bind a slot to a term. One write, no training."""
        if term is None:
            return
        self.slots.write(self.slot_key[slot], self.lex.code(term))
        self.note_mention(term)

    def note_mention(self, term):
        if term in self.mentioned:
            self.mentioned.remove(term)
        self.mentioned.insert(0, term)
        del self.mentioned[64:]

    def recall(self, slot, top=1, floor=0.25):
        """What is in a slot: the pattern read out, then named by whichever mentioned term it overlaps most."""
        pattern = self.slots.read(self.slot_key[slot])
        hits = [h for h in self.lex.nearest(pattern, self.mentioned, top=top) if h[1] >= floor]
        return hits[0] if top == 1 and hits else (hits if top != 1 else None)

    # -------------------------------------------------------------------- turns
    def advance(self, text, kind="said"):
        """A new turn: bind what was said to this moment, so it can be recalled by time as well as by content."""
        self.turn += 1
        code = self.lex.bag(text.lower().split()) if text else torch.zeros(self.n)
        self.episodes.write(self.time(self.turn), code)
        self.history.append((self.turn, text, kind))
        return code

    def at_turn(self, t, top=3):
        """What was said at a given turn, named from the pattern - memory, not a transcript lookup."""
        if t < 1 or t > self.turn:
            return []
        return self.lex.nearest(self.episodes.read(self.time(t)), self.mentioned, top=top)

    def last_turns(self, n=2):
        return self.history[-n:]

    def clear(self):
        self.slots.clear()
        self.mentioned.clear()

    def synapses(self):
        return self.slots.synapses() + self.episodes.synapses()

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/agent.py
"""The turn loop, which is the brain.

Every stage of a turn is one of the mechanisms, in the order the tissue would do it. Nothing here is a wrapper
around a lookup table, and no stage exists only to make the others reachable:

  1. lexical cortex    the sentence becomes sparse activity (codes.Lexicon). Nothing enters any other way.
  2. working memory    the utterance is bound to this turn; pronouns are resolved by reading prefrontal slots,
                       not by a rule about the previous sentence.
  3. situation code    what was asked *and* what each store currently has, as one sparse pattern.
  4. the gate          basal-ganglia selection: experts vote on which system answers, and learn from whether it
                       produced grounded evidence. A cascade of ifs it is not.
  5. the system acts   the cognitive map infers, the episodic store completes a pattern, the semantic store
                       reports background structure.
  6. neuromodulation   surprise - the gap between what the map predicted and what was said - sets how strongly
                       the fact is written. Expected facts leave a faint trace, contradictions a strong one.
  7. consolidation     when the conversation goes quiet, replay: the episodic store proposes, the map relaxes,
                       weak experts are pruned. Nothing is trained by gradient anywhere in this file.
"""
import os
import re
import sys

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
for p in (os.path.join(ROOT, "chat"), ROOT):
    if p not in sys.path:
        sys.path.insert(0, p)

from brain import speech                 # noqa: E402
from brain.codes import Lexicon          # noqa: E402
from brain.episodic import Episodes      # noqa: E402
from brain.gate import Gate              # noqa: E402
from brain.semantic import Semantic      # noqa: E402
from brain.tracts import Tracts          # noqa: E402
from brain.working import Prefrontal     # noqa: E402
import language as L                     # noqa: E402  the hand-written parser: no pretrained weights anywhere
from mapsystem import Brain as MapSystem  # noqa: E402  the cognitive map and its query layer

PRONOUNS = {"she": "topic", "he": "topic", "it": "topic", "they": "topic", "her": "object", "him": "object",
            "them": "object", "that": "topic", "this": "topic", "those": "object", "there": "topic",
            "the same": "topic"}
RECALL = (re.compile(r"^(what|who) (did|have) (we|i|you) (just )?(talk|say|said|discuss)", re.I),
          re.compile(r"^what did (i|you) say about (.+)", re.I),
          re.compile(r"^(do you )?remember", re.I),
          re.compile(r"^what (was|were) (we|i|you) (talking|saying)", re.I))
GREET = re.compile(r"^(hi|hey|hello|good (morning|evening|afternoon)|yo)\b", re.I)
PERSONAL = re.compile(r"\b(i|my|me|we|our)\b", re.I)
SLEEP_EVERY = 6      # quiet-period consolidation: after this many new facts, replay before the next answer


class Agent:
    def __init__(self, seed=0, background=True):
        self.lex = Lexicon(seed=seed)
        self.wm = Prefrontal(self.lex)
        self.epi = Episodes(self.lex)
        self.gate = Gate(self.lex)
        self.maps = MapSystem()                       # hippocampal map: one space per relation kind
        self.sem = Semantic() if background else Semantic(path=None)
        self.tracts = Tracts()                        # how long each pathway takes, and when the gate must choose
        self.turns = 0
        self.unslept = 0
        self.sleep_every = SLEEP_EVERY
        self.trace = []           # per turn: what the gate saw, chose, and why - the panel reads this

    # ------------------------------------------------------------------ a turn
    def say(self, text):
        out = []
        for sentence in (L.sentences(text) or [text]):
            out.append(self._turn(sentence))
        return out

    def _turn(self, sentence):
        self.turns += 1
        self.wm.advance(sentence)
        p = self._understand(sentence)
        signals, arrived = self._signals(p)
        allowed = self._applicable(p, sentence, arrived)
        action, conf, how, code = self.gate.choose(p["intent"], signals, allowed)
        result = self._act(action, p, sentence)
        if result["grounded"]:
            self.gate.reinforce(code, action)
        else:
            for alt in (x for x in self._alternatives(action) if x in allowed):
                other = self._act(alt, p, sentence)
                if other["grounded"]:
                    self.gate.correct(code, action, alt)   # match tracking: this situation gets its own expert
                    action, result = alt, other
                    break
            else:
                # no store had anything. Saying so is the right move here, and the gate learns that too, so a
                # situation that has never grounded stops being sent round every system first.
                action, result = "clarify", self._do_clarify(p, sentence)
                self.gate.reinforce(code, action)
        self._remember(p, result, sentence)
        self.epi.store(sentence, self.turns, p["intent"], payload={"action": action})
        step = {"turn": self.turns, "sentence": sentence, "intent": p["intent"], "action": action,
                "via": how, "confidence": round(conf, 3), "signals": [s for s, on in signals.items() if on],
                "grounded": result["grounded"], "reply": result["reply"], "evidence": result.get("evidence"),
                "arrived": [r for r, ok in arrived.items() if ok],
                "decided_at": self.tracts.deadline, "timeline": self.tracts.timeline()}
        self.trace.append(step)
        if self.unslept >= self.sleep_every:
            self.sleep(quiet=True)
        return step

    # -------------------------------------------------------------- understanding
    def _understand(self, sentence):
        """Parse, then resolve what the sentence points at using working memory rather than word order."""
        p = dict(L.parse(sentence))
        p["resolved"] = {}
        p["unresolved"] = False
        for field in ("a", "b"):
            if not p.get(field):
                continue
            key, shown = p[field]
            slot = PRONOUNS.get(key)
            if slot is None:
                continue
            hit = self.wm.recall(slot) or self.wm.recall("topic")
            if hit:
                p[field] = (hit[0], self.maps._say(hit[0]) if hit[0] in self.maps.shown else hit[0])
                p["resolved"][shown] = {"term": hit[0], "strength": round(hit[1], 3), "slot": slot}
            else:
                p["unresolved"] = True
        low = sentence.strip().lower()
        if p["kind"] == "ask":
            p["intent"] = p["q"]
        elif p["kind"] == "fact":
            p["intent"] = "fact"
        elif p["kind"] == "command":
            p["intent"] = "command"
        elif any(r.search(low) for r in RECALL):
            p["intent"] = "recall"
        elif GREET.match(low) or low in ("thanks", "thank you", "bye", "goodbye"):
            p["intent"] = "social"
        elif PERSONAL.search(low):
            p["intent"] = "social"        # something about the speaker, not a relation between things
        else:
            p["intent"] = "unknown"
        return p

    def _space(self, p):
        """Which map a turn is about. Space is the default for questions that name no dimension ("where is the
        vet from the cafe"); "between" belongs to whichever map actually holds both things."""
        if p.get("axis"):
            return L.space_of(p["axis"])
        if p.get("moves"):
            return L.space_of(p["moves"][0][0])
        if p.get("q") in ("where", "direction", "at"):
            return "place"
        if p.get("q") == "between" and p.get("a") and p.get("b"):
            for space in self.maps.maps:
                if self.maps._connected(space, p["a"][0], p["b"][0]):
                    return space
        return None

    def _signals(self, p):
        """What each store has to offer, and which of them got a signal here in time.

        A store's contribution is gated by its conduction delay: the gate chooses at a deadline, and anything
        still travelling is simply absent from the situation code. That is what makes the order of the turn a
        physical fact rather than the order these lines happen to be written in.
        """
        a = p["a"][0] if p.get("a") else None
        b = p["b"][0] if p.get("b") else None
        space = self._space(p)
        arrived = {r: self.tracts.in_time("lexical", r) for r in ("working", "episodic", "map", "semantic")}
        known = lambda t: bool(t) and any(m.knows(t) for m in self.maps.maps.values())
        sig = {"knows_a": arrived["map"] and known(a), "knows_b": arrived["map"] and known(b),
               "connected": bool(arrived["map"] and space and a and b and self.maps._connected(space, a, b)),
               "space_exists": bool(arrived["map"] and space and space in self.maps.maps),
               "semantic_a": bool(arrived["semantic"] and a and self.sem.knows(a)),
               "topic_held": arrived["working"] and self.wm.recall("topic") is not None,
               "resolved_ref": bool(p.get("resolved")),
               "unresolved_ref": bool(p.get("unresolved")),
               "episodes": arrived["episodic"] and len(self.epi.records) > 0,
               "facts_held": arrived["map"] and len(self.maps.log) > 0,
               "late_map": not arrived["map"]}
        return sig, arrived

    # ------------------------------------------------------------------- acting
    def _applicable(self, p, sentence, arrived=None):
        """What this turn could be answered by at all - a type check, not a routing decision. A statement cannot
        be queried and a question cannot be stored. A store whose signal has not arrived by the deadline is out
        too, for a different reason: it has nothing to say yet."""
        arrived = arrived or {r: True for r in ("working", "episodic", "map", "semantic")}
        kind, intent = p["kind"], p["intent"]
        if kind == "command":
            return ["social"]
        if kind == "fact":
            return ["teach", "clarify"]
        if kind == "ask" and hasattr(self.maps, "_ask_" + p.get("q", "")):
            out = ["infer", "clarify"] if arrived["map"] else ["recall", "clarify"]
            if p["q"] == "about":                      # "tell me about X" - the stores can compete for this
                out = [a for a in ("describe", "infer", "recall") if a != "infer" or arrived["map"]] + ["clarify"]
            return out
        out = ["clarify"]
        if arrived["episodic"] and (intent == "recall" or (self.epi.records and self._topic_of(sentence))):
            out.insert(0, "recall")
        if self._topic_of(sentence):
            out.insert(0, "describe")
        if intent == "social" or GREET.match(sentence.strip().lower()):
            out.insert(0, "social")
        return out

    def _alternatives(self, action):
        """Who else gets a try when the selected system comes back empty, best-placed first. Clarifying is not
        in any list: it produces no evidence, so it is where the turn ends rather than something to try."""
        order = {"teach": ("infer", "describe", "recall", "social"),
                 "infer": ("teach", "recall", "describe", "social"),
                 "recall": ("infer", "describe", "social"),
                 "describe": ("infer", "recall", "social"),
                 "social": ("recall", "describe", "infer"),
                 "clarify": ("infer", "teach", "recall", "describe", "social")}
        return tuple(x for x in order.get(action, ()) if x != action)

    def _act(self, action, p, sentence):
        return getattr(self, "_do_" + action)(p, sentence)

    def _do_teach(self, p, sentence):
        if p["kind"] != "fact":
            return {"grounded": False, "reply": speech.clarify("unparsed")}
        surprise, note = self._surprise(p)
        space = self._space(p)
        self.maps.log.append(sentence)
        # Neuromodulation: surprise sets how hard this fact is written *now* - an expected fact barely moves
        # anything, an unexpected one is encoded strongly. It does not change how loudly the fact votes later:
        # consolidation weighs facts by how many others agree, which is what keeps one loud mistake from winning.
        gain = 0.5 + 1.5 * surprise
        m = self.maps.maps.get(space)
        if m is not None:
            was, m.lr = m.lr, min(1.0, m.lr * gain)
            reply = self.maps._teach(p)
            m.lr = was
        else:
            reply = self.maps._teach(p)
        self.unslept += 1
        return {"grounded": True, "reply": reply + note,
                "evidence": {"surprise": round(surprise, 3), "gain": round(gain, 3), "space": space}}

    def _surprise(self, p):
        """Prediction error before the fact is stored: 0 if the map already implied it, 1 if it contradicts.

        Dopamine and acetylcholine decide how strongly something is written down, and what they respond to is
        the gap between prediction and outcome. Nothing was gating encoding here before; everything was learned
        with the weight the corpus happened to give it.
        """
        space = self._space(p)
        a, b = p["a"][0], p["b"][0]
        if not space or space not in self.maps.maps or not self.maps._connected(space, b, a):
            return 0.6, ""                     # new material: worth encoding, nothing to contradict
        self.maps._settle(space)
        predicted = self.maps._offset(space, b, a)
        worst = 0.0
        for axis, c in p["moves"]:
            have = predicted.get(axis, 0.0)
            agreement = (c * have) / (abs(c) * max(abs(have), 1e-6))
            worst = max(worst, (1.0 - agreement) / 2.0)
        return worst, ""

    def _do_infer(self, p, sentence):
        if p["kind"] != "ask" or not hasattr(self.maps, "_ask_" + p.get("q", "")):
            return {"grounded": False, "reply": speech.clarify("unparsed")}
        self.maps._asked = {x[0]: x[1] for x in (p.get("a"), p.get("b")) if x}
        space = self._space(p)
        a = p["a"][0] if p.get("a") else None
        b = p["b"][0] if p.get("b") else None
        reply = getattr(self.maps, "_ask_" + p["q"])(p)
        answered = bool(space and space in self.maps.maps and (
            (a and b and self.maps._connected(space, a, b)) or
            (p["q"] in ("extreme", "rank") and space in self.maps.maps) or
            (p["q"] == "at" and a and self.maps.maps[space].knows(a))))
        # A justified "I can't tell you" is an answer too: never heard of it, or heard of both and nothing links
        # them. The map knows which of those it is, and no other store can do better - handing the question on
        # anyway is how the old router produced confident irrelevance.
        nowhere = any(t and not any(m.knows(t) for m in self.maps.maps.values()) and not self.sem.knows(t)
                      for t in (a, b))
        both_known = bool(space and space in self.maps.maps and a and b
                          and self.maps.maps[space].knows(a) and self.maps.maps[space].knows(b))
        grounded = (answered or nowhere or both_known) and not p.get("unresolved")
        return {"grounded": grounded, "reply": reply,
                "evidence": {"space": space, "steps": self._steps(space, a, b, p)}}

    def _steps(self, space, a, b, p):
        """The number the answer rests on, so a reply is never a claim without its grounds."""
        axis = p.get("axis")
        if not (space and axis and a and b) or space not in self.maps.maps:
            return None
        m = self.maps.maps[space]
        if not (m.knows(a) and m.knows(b) and axis in m.rel):
            return None
        return round(m.steps(b, axis, a), 3)

    def _do_recall(self, p, sentence):
        """The episodic store answers: pattern completion from whatever the question happens to say."""
        cue = sentence
        m = re.search(r"about (.+)", sentence, re.I)
        if m:
            cue = m.group(1).strip(" ?.")
        # "what did we talk about" names nothing to complete from - it is a request for the last thing that
        # carried content, not a cue. Completing on its own words lands on whatever question sounds like it.
        generic = not m and p["intent"] == "recall"
        record, conf = (None, 0.0) if generic else self.epi.recall(cue)
        if record is not None and record["turn"] == self.turns:   # never answer with the question itself
            record, conf = None, 0.0
        if record is None or conf < 0.3:
            # "what did we just talk about" has no content to complete from; the last turn is the honest answer
            older = [r for r in self.epi.records if r["turn"] < self.turns and r["kind"] != "recall"]
            if not older or p["intent"] != "recall":
                return {"grounded": False, "reply": "I don't have anything remembered that matches that."}
            # "what did we talk about" has no content to complete from, so the honest answer is the last thing
            # that carried any - a fact if one was said, otherwise whatever was last said
            facts = [r for r in older if r["kind"] == "fact"]
            record, conf = (facts or older)[-1], 0.6   # not a faint memory: the last thing that carried content
        return {"grounded": True,
                "reply": speech.recalled(record, conf, self.turns - record["turn"]),
                "evidence": {"turn": record["turn"], "match": round(conf, 3)}}

    def _do_describe(self, p, sentence):
        term = (p["a"][0] if p.get("a") else None) or self._topic_of(sentence)
        if not term:
            return {"grounded": False, "reply": speech.clarify("unparsed")}
        if any(m.knows(term) for m in self.maps.maps.values()):
            self.maps._asked = {term: term}
            return {"grounded": True, "reply": self.maps._ask_about({"a": (term, term)}),
                    "evidence": {"store": "map"}}
        detail = self.sem.describe(term)
        if detail and (detail["relations"] or detail["near"]):
            return {"grounded": True, "reply": speech.about_semantic(term, detail),
                    "evidence": {"store": "semantic", "near": [n for n, _ in detail["near"][:3]]}}
        return {"grounded": False, "reply": speech.clarify("unknown-term", term)}

    def _topic_of(self, sentence):
        words = [w for w in re.findall(r"[a-zA-Z][a-zA-Z'-]+", sentence.lower()) if len(w) > 2]
        for w in words:
            if any(m.knows(w) for m in self.maps.maps.values()) or self.sem.knows(w):
                return w
        return None

    def _do_clarify(self, p, sentence):
        """Asking back. Never grounded - it rests on no store - so it is only ever where a turn ends."""
        if p.get("unresolved"):
            return {"grounded": False, "reply": speech.clarify("pronoun")}
        term = self._topic_of(sentence)
        if term and not any(m.knows(term) for m in self.maps.maps.values()):
            return {"grounded": False, "reply": speech.clarify("unknown-term", term)}
        return {"grounded": False, "reply": speech.clarify("unparsed")}

    def _do_social(self, p, sentence):
        low = sentence.strip().lower()
        if p["kind"] == "command":
            if p["do"] == "sleep":
                return {"grounded": True, "reply": self.sleep()}
            if p["do"] == "reset":
                self.forget()
                return {"grounded": True, "reply": "Forgotten - every store is empty again."}
            if p["do"] == "summary":
                return {"grounded": True, "reply": self.summary()}
            return {"grounded": True, "reply": self.maps._cmd_help()}
        if GREET.match(low) or low in ("thanks", "thank you"):
            return {"grounded": True, "reply": speech.greeting(len(self.maps.shown), len(self.maps.log))}
        topic = self._topic_of(sentence)
        said = None
        if topic:
            hits = self.epi.about(topic, limit=2)
            older = [(r, s) for r, s in hits if r["turn"] < self.turns]
            if older:
                said = f'you said "{older[0][0]["text"]}"'
        # being spoken to is always answerable: acknowledging, and saying plainly what is and isn't held, is a
        # real reply rather than a failure to route
        return {"grounded": True, "reply": speech.acknowledge(topic, said), "evidence": {"topic": topic}}

    # ---------------------------------------------------------------- memory
    def _remember(self, p, result, sentence):
        """What prefrontal cortex holds after this turn. Written, not appended to a list."""
        a = p["a"][0] if p.get("a") else None
        b = p["b"][0] if p.get("b") else None
        if a:
            self.wm.hold("subject", a)
            self.wm.hold("topic", a)
        if b:
            self.wm.hold("object", b)
        if not a and not b:
            topic = self._topic_of(sentence)
            if topic:
                self.wm.hold("topic", topic)
        if p.get("axis"):
            self.wm.hold("relation", p["axis"])
        self.wm.hold("answer", result["reply"][:40])

    # ---------------------------------------------------------------- offline
    def sleep(self, quiet=False):
        """Consolidation. The episodic store proposes what to replay, the map relaxes under all of it at once,
        and experts that have stopped winning are let go."""
        moved = 0.0
        for space, m in self.maps.maps.items():
            before = m.pos.clone()
            m.sleep(self.maps.config["map_config"]["sleep"], tolerance=self.maps.config["map_config"]["tol"])
            delta = m.pos - before
            if len(delta):
                delta = delta - delta.mean(0)
                moved = max(moved, float(delta.norm(dim=1).max()))
        self.maps.dirty.clear()
        self.unslept = 0
        replayed = self.epi.replay(8)
        return speech.sleep_report(sum(len(m.episodes) for m in self.maps.maps.values()), moved, len(self.gate.pool))

    def forget(self):
        self.maps.reset()
        self.wm.clear()
        self.epi = Episodes(self.lex)
        self.unslept = 0

    # ------------------------------------------------------------------ status
    def summary(self):
        g = self.gate.stats()
        facts = sum(len(m.episodes) for m in self.maps.maps.values())
        return (f"{facts} fact(s) on {len(self.maps.maps)} map(s), {len(self.epi.records)} episode(s) remembered, "
                f"{g['experts']} gate expert(s) grown from use ({g['born']} born), "
                f"{self.footprint():,} synapses in all.")

    def footprint(self):
        return (self.wm.synapses() + self.epi.synapses() + self.gate.pool.synapses()
                + sum(m.footprint() for m in self.maps.maps.values()))

    def state(self):
        return {"turns": self.turns, "facts": len(self.maps.log), "episodes": len(self.epi.records),
                "gate": self.gate.stats(), "synapses": self.footprint(), "tracts": self.tracts.spec(),
                "columns": [self.lex.column.spec(), self.epi.column.spec()],
                "held": {s: self.wm.recall(s) for s in ("topic", "subject", "object", "relation")},
                "maps": self.maps.view(), "trace": self.trace[-12:]}

In [ ]:
%%writefile /kaggle/working/neuromoe/tests/check_brain.py
"""Checks that the brain mechanisms are *in the loop*, not sitting beside it.

Each check names the mechanism it is evidence for. If a mechanism were replaced by a lookup table or an if-else,
the corresponding check should fail - that is what they are for.
"""
import os
import sys

import torch

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, ROOT)

from brain.agent import Agent  # noqa: E402
from brain.codes import Lexicon  # noqa: E402

ok = True


def check(name, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"[{'PASS' if cond else 'FAIL'}] {name}{'  -> ' + str(detail)[:140] if detail else ''}")


def reply(a, text):
    return a.say(text)[-1]["reply"]


def step(a, text):
    return a.say(text)[-1]


# 1. lexical cortex: sparse codes, similarity by overlap, unbounded vocabulary -----------------------------
lex = Lexicon(seed=0)
dog, dogs, violin = lex.code("dog"), lex.code("dogs"), lex.code("violin")
check("lexical code is sparse and exactly k units", float(dog.sum()) == lex.k and dog.max() == 1.0)
check("morphology overlaps, unrelated words do not", float(dog @ dogs) > float(dog @ violin),
      f"dog.dogs={float(dog @ dogs):.0f} vs dog.violin={float(dog @ violin):.0f}")
check("a word never seen before still has a code", float(lex.code("zzxqv").sum()) == lex.k)

a = Agent(background=False)

# 2. cognitive map: one-shot facts, inference never stated -------------------------------------------------
a.say("Dan is taller than Eve. Alice is taller than Bob. Cara is taller than Dan. Bob is taller than Cara.")
r = reply(a, "Is Alice taller than Eve?")
check("map infers a 4-step chain told out of order", r.startswith("Yes"), r)

# 3. prefrontal working memory: a pronoun resolved from a bound slot, not from word order -------------------
s = step(a, "Is she taller than Cara?")
check("pronoun resolved from working memory", s["reply"].startswith("Yes") and "resolved_ref" in s["signals"],
      s["reply"])
a2 = Agent(background=False)
s = step(a2, "Is she taller than Bob?")
check("pronoun with nothing bound asks instead of guessing", "mean" in s["reply"].lower(), s["reply"])

# 4. episodic store: pattern completion over what was said -------------------------------------------------
s = step(a, "what did we just talk about")
check("episodic recall returns something actually said", s["action"] == "recall" and "said" in s["reply"],
      s["reply"])
rec, conf = a.epi.recall("Cara taller")
check("a partial cue completes to the right episode", rec is not None and "Cara" in rec["text"],
      f"{rec and rec['text']} ({conf:.2f})")

# 5. basal-ganglia gate: grows experts from use and learns to select ----------------------------------------
before = a.gate.stats()["experts"]
for q in ("Who is the tallest?", "Is Bob taller than Dan?", "Rank everyone by height"):
    a.say(q)
after = a.gate.stats()
check("gate recruits experts from use (neurogenesis)", after["experts"] > 0 and after["born"] > 0, after)
check("gate takes over from the reflexes", any(t["via"] == "learned" for t in a.trace),
      [t["via"] for t in a.trace][-6:])
check("gate selection is recorded per turn", set(a.trace[-1]) >= {"action", "via", "signals", "grounded"})

# 6. the gate corrects itself when a system comes back empty ------------------------------------------------
a3 = Agent(background=False)
a3.say("Alice is taller than Bob.")
s = step(a3, "Is Zog taller than Alice?")
check("no store has it: says so rather than inventing", "don't know" in s["reply"].lower(), s["reply"])

# 7. neuromodulation: surprise gates how strongly a fact is written ------------------------------------------
agree = step(a3, "Alice is taller than Bob.")["evidence"]
clash = step(a3, "Bob is taller than Alice.")["evidence"]
check("a contradiction is more surprising than a repetition", clash["surprise"] > agree["surprise"] + 0.3,
      f"{agree['surprise']:.2f} vs {clash['surprise']:.2f}")
check("surprise scales how hard the fact is written", clash["gain"] > agree["gain"],
      f"gain {agree['gain']:.2f} -> {clash['gain']:.2f}")
check("surprise does not buy a louder vote in consolidation",
      set(a3.maps.maps["height"].weights) == {1.0}, a3.maps.maps["height"].weights)

# 8. sleep: replay settles the map and outvotes the lone contradiction ---------------------------------------
a.say("Eve is taller than Alice.")
a.sleep()
check("one contradicted fact is outvoted after replay", reply(a, "Is Alice taller than Eve?").startswith("Yes"))

# 9. the memory grows with use: synaptogenesis, not a fixed parameter count ----------------------------------
small = Agent(background=False)
base = small.footprint()
small.say("Ann is older than Ben. Ben is older than Cal.")
grown = small.footprint()
check("footprint grows as it is used", grown > base, f"{base:,} -> {grown:,}")

# 10. nothing anywhere is trained by gradient ----------------------------------------------------------------
tensors = [small.gate.pool.P, small.gate.pool.O, small.wm.slots.M, small.epi.codes]
tensors += [m.pos for m in small.maps.maps.values()]
check("no tensor requires a gradient", not any(t.requires_grad for t in tensors))

print("\nALL PASS" if ok else "\nFAILURES ABOVE")
sys.exit(0 if ok else 1)

In [ ]:
%%writefile /kaggle/working/neuromoe/tests/check_tissue.py
"""Checks for the three tissue properties: conduction delay, non-linear summation, and one repeated microcircuit.

Each check is meant to fail if the property is decorative. A delay that changes no decision, a dendrite that
sums linearly and a "column" that is a different machine in each area would all be caught here.
"""
import os
import sys

import torch

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, ROOT)

from brain.agent import Agent  # noqa: E402
from brain.codes import Lexicon  # noqa: E402
from brain.column import Column  # noqa: E402
from brain.dendrite import Dendrites  # noqa: E402
from brain.tracts import Tracts  # noqa: E402

ok = True


def check(name, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"[{'PASS' if cond else 'FAIL'}] {name}{'  -> ' + str(detail)[:150] if detail else ''}")


# 1. sequentiality: distance makes order, and order has consequences --------------------------------------
t = Tracts()
check("nothing arrives instantly", all(t.delay("lexical", r) >= 1 for r in t.layout if r != "lexical"))
check("the nearer store is heard from first", t.arrival("lexical", "working") < t.arrival("lexical", "map"),
      f"working {t.arrival('lexical', 'working')} ticks, map {t.arrival('lexical', 'map')}")
check("relayed signals take the sum of their hops",
      t.arrival("lexical", "gate", via=["working"]) == t.delay("lexical", "working") + t.delay("working", "gate"))

a = Agent(background=False)
a.say("Alice is taller than Bob. Bob is taller than Cara.")
slow = Agent(background=False)
slow.say("Alice is taller than Bob. Bob is taller than Cara.")
slow.tracts.deadline = 3          # urgency: the map's signal is still in transit when the gate must choose
calm = a.say("Is Alice taller than Cara?")[-1]
hasty = slow.say("Is Alice taller than Cara?")[-1]
check("with time to wait, the map answers", calm["reply"].startswith("Yes") and "map" in calm["arrived"],
      calm["reply"])
check("under urgency the map is not there yet, and the turn says so",
      "map" not in hasty["arrived"] and hasty["action"] != "infer", f"{hasty['action']}: {hasty['reply'][:60]}")
check("the deadline is what changed, not the knowledge",
      slow.maps.maps["height"].knows("alice") and slow.tracts.deadline < t.arrival("lexical", "map"))

# 2. non-linear summation and suppression ------------------------------------------------------------------
d = Dendrites(64, n=512, k=16, fan_in=8, branches=4, seed=1, theta=1.0, divisive=0.0)
unit = 0
b0 = d.idx[unit, 0]                                   # two synapses on the same branch
spread = (d.idx[unit, 0, 0].item(), d.idx[unit, 1, 0].item())   # one synapse on each of two branches


def drive_of(inputs):
    x = torch.zeros(1, 64)
    for i in inputs:
        x[0, i] = 1.0
    s = (x[:, d.idx] * d.w).sum(-1).clamp_min(0)
    return float((s * s / (d.theta + s)).sum(-1)[0, unit])


together = drive_of([int(b0[0]), int(b0[1])])
apart = drive_of(list(spread))
single = drive_of([int(b0[0])])
check("two inputs on one branch beat the same two spread across branches", together > apart,
      f"same branch {together:.3f} vs spread {apart:.3f}")
check("summation within a branch is supralinear, not additive", together > 2 * single,
      f"{together:.3f} vs 2x{single:.3f}")

quiet = Dendrites(64, n=512, k=16, seed=2, divisive=0.0)
loud = Dendrites(64, n=512, k=16, seed=2, divisive=0.5)
x = torch.zeros(1, 64)
x[0, :8] = 1.0
s = (x[:, loud.idx] * loud.w).sum(-1)
branch = s * s / (loud.theta + s)
raw = branch.sum(-1)
shunted = raw / (1.0 + loud.divisive * raw.mean(dim=1, keepdim=True))
check("divisive inhibition compresses the range instead of clipping it",
      float(shunted.max() / shunted.min().clamp_min(1e-6)) < float(raw.max() / raw.min().clamp_min(1e-6)),
      f"raw {float(raw.max() / raw.min().clamp_min(1e-6)):.1f}x -> shunted "
      f"{float(shunted.max() / shunted.min().clamp_min(1e-6)):.1f}x")
strong = loud(x * 10.0)
check("a ten times louder input still fires exactly k units", int(strong.sum()) == loud.k)

fatigued = Dendrites(64, n=512, k=16, seed=3, adapt=0.5)
first = fatigued(x).clone()
for _ in range(6):
    fatigued(x)
after = fatigued(x)
check("repeating the same input suppresses the branches that carried it",
      float(fatigued.fatigue.max()) > 0 and float((first * after).sum()) <= float(first.sum()),
      f"overlap {float((first * after).sum()):.0f} of {float(first.sum()):.0f}, "
      f"max fatigue {float(fatigued.fatigue.max()):.2f}")

# 3. one microcircuit, repeated -------------------------------------------------------------------------
lex = Lexicon(seed=0)
agent = Agent(background=False)
areas = {"lexical": agent.lex.column, "episodic": agent.epi.column}
check("every area is the same microcircuit", all(isinstance(c, Column) for c in areas.values()),
      {k: type(v).__name__ for k, v in areas.items()})
specs = {k: c.spec() for k, c in areas.items()}
check("areas differ in size and tuning, not in kind",
      len({tuple(sorted(s.keys())) for s in specs.values()}) == 1
      and specs["lexical"]["units"] != specs["episodic"]["units"],
      {k: (s["units"], s["active"], s["branches"]) for k, s in specs.items()})
c = Column("test", 32, n=256, k=8, branches=2, n_out=3, seed=0)
code = c.encode(torch.eye(32)[0])
c.associate(code, 2)
check("a column can learn an association locally", int(c.read(code).argmax()) == 2)
check("column activity is sparse everywhere", int(code.sum()) == c.k)

print("\nALL PASS" if ok else "\nFAILURES ABOVE")
sys.exit(0 if ok else 1)

In [ ]:
%%writefile /kaggle/working/neuromoe/tests/check_chat.py
"""End-to-end checks for the chat interface: controlled English in, answers read off the cognitive maps out."""
import os
import sys

sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "chat"))
import language as L  # noqa: E402
from mapsystem import Brain  # noqa: E402

ok = True


def check(name, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"[{'PASS' if cond else 'FAIL'}] {name}{'  -> ' + detail if detail else ''}")


def ask(b, text):
    return b.say(text)[-1]["reply"]


# parser ------------------------------------------------------------------------------------------------
P = L.parse
check("fact: comparative", P("Alice is taller than Bob.")["moves"] == [("height", 1)])
check("fact: opposite comparative", P("Bob is shorter than Alice")["moves"] == [("height", -1)])
check("fact: more X than", P("Cara is more careful than Dan")["moves"] == [("careful", 1)])
check("fact: compound direction", P("The bank is north-east of the park")["moves"] == [("north", 1), ("east", 1)])
check("fact: counted direction", P("Dan is 2 blocks east of Eve")["moves"] == [("east", 2)])
check("fact: left of", P("The lamp is left of the bed")["moves"] == [("east", -1)])
check("names drop articles", P("The Cafe is north of the park")["a"] == ("cafe", "The Cafe"))
check("question: compare", P("Is Alice taller than Cara?")["q"] == "compare")
check("question: which", P("Who is taller, Alice or Dan?")["q"] == "which")
check("question: superlative", P("Who is the tallest?")["axis"] == "height")
check("question: superlative -iest", P("Who is the happiest?")["axis"] == "happiness")
check("question: rank", P("Rank everyone by height")["q"] == "rank")
check("question: at", P("What is 2 blocks north of the park?")["moves"] == [("north", 2)])
check("question: direction", P("Is the bank north-east of the park?")["q"] == "direction")
check("unknown sentence refused", P("The weather is lovely today")["kind"] == "unknown")
check("sentence splitting", len(L.sentences("A is taller than B. B is taller than C! Is A taller than C?")) == 3)

# transitive inference over a shuffled chain, told once each --------------------------------------------
b = Brain()
b.say("Dan is taller than Eve. Alice is taller than Bob. Cara is taller than Dan. Bob is taller than Cara.")
r = ask(b, "Is Alice taller than Eve?")
check("transitive: 4-step inference", r.startswith("Yes"), r)
r = ask(b, "Is Eve taller than Bob?")
check("transitive: reversed", r.startswith("No"), r)
r = ask(b, "Who is the tallest?")
check("superlative", r.startswith("Alice"), r)
r = ask(b, "Who is the shortest?")
check("superlative, other direction", r.startswith("Eve"), r)
r = ask(b, "Rank everyone by height")
check("ranking", "Alice > Bob > Cara > Dan > Eve" in r, r)
r = ask(b, "Who is between Bob and Eve?")
check("between", "Cara" in r and "Dan" in r and "Alice" not in r, r)
r = ask(b, "How much taller is Alice than Eve?")
check("how much", "4 steps" in r, r)
r = ask(b, "Is Alice older than Bob?")
check("different dimension: doesn't know", "don't know" in r.lower(), r)
r = ask(b, "Is Zed taller than Bob?")
check("unknown item", "don't know anything about Zed" in r, r)

# islands: unknown until linked, then known ---------------------------------------------------------------
b.say("Xavier is taller than Yuri.")
r = ask(b, "Is Xavier taller than Alice?")
check("unlinked islands -> I don't know", "don't know" in r.lower(), r)
b.say("Eve is taller than Xavier.")
r = ask(b, "Is Alice taller than Yuri?")
check("islands linked by one fact", r.startswith("Yes"), r)

# contradiction is flagged and outvoted --------------------------------------------------------------------
r = ask(b, "Eve is taller than Alice.")
check("contradiction noticed", "contradicts" in r, r)
b.say("sleep")
r = ask(b, "Is Alice taller than Eve?")
check("lone contradiction outvoted after sleep", r.startswith("Yes"), r)

# space: routes and retrieval by location ------------------------------------------------------------------
s = Brain()
s.say("The cafe is north of the park. The bank is east of the cafe. The school is 2 blocks east of the park. "
      "The library is north of the bank.")
r = ask(s, "What is 1 block north and 1 east of the park?")
check("retrieval by location (never told directly)", r.startswith("the bank") or r.startswith("The bank"), r)
r = ask(s, "Where is the library relative to the park?")
check("path integration", "2 north" in r and "1 east" in r, r)
r = ask(s, "Is the school south-east of the cafe?")
check("direction check", r.startswith("Yes"), r)
r = ask(s, "Is the park north of the bank?")
check("direction check (no)", r.startswith("No"), r)
r = ask(s, "What is 5 blocks west of the park?")
check("empty location", "Nothing I know of" in r, r)
r = ask(s, "Who is between the park and the school?")
check("spatial between (nothing there)", "Nothing" in r, r)
s.say("The shop is east of the park.")
r = ask(s, "What is between the park and the school?")
check("spatial between", "shop" in r, r)

r = ask(s, "what do you know")
check("summary", "Places" in r, r.replace("\n", " | "))
check("view has points", s.view()[0]["kind"] == "place" and len(s.view()[0]["points"]) == 6)

print("\nALL PASS" if ok else "\nFAILURES ABOVE")
sys.exit(0 if ok else 1)

In [ ]:
%%writefile /kaggle/working/neuromoe/tests/check_cogmap.py
"""Correctness checks for the pseudospatial memory (cogmap.py)."""
import os
import random
import sys

import numpy as np
import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import map_bench as mb
from cogmap import CognitiveMap

ok = True


def check(name, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"[{'PASS' if cond else 'FAIL'}] {name}{'  ' + detail if detail else ''}")


# 1. one-shot placement: the first relation fixes the new item's position exactly
m = CognitiveMap(dim=2)
m.observe("A", "above", "B")
check("one fact places the new item one step away", abs(m.steps("A", "above", "B") - 1.0) < 1e-6)

# 2. transitive inference from adjacent pairs only, shuffled, clean: every unseen pair right, full ranking right
items = [f"i{k}" for k in range(20)]
pairs = [(items[k], items[k + 1]) for k in range(19)]
random.Random(1).shuffle(pairs)
m = CognitiveMap(dim=2, robust=0.3)
for lo, hi in pairs:
    m.observe(lo, "above", hi)
m.sleep(300)
acc = np.mean([(m.steps(items[i], "above", items[j]) > 0) == (j > i)
               for i in range(20) for j in range(20) if abs(i - j) > 1])
check("clean transitive inference on never-seen pairs", acc == 1.0, f"{acc:.3f}")
check("full ranking recovered", m.ranking("above") == items)
check("items between two others are found", set(m.between("i3", "i7", "above")) == {"i4", "i5", "i6"})

# 3. re-centering moves the map but changes no relation
before = m.steps("i2", "above", "i15")
m.recenter()
check("re-centering preserves every relation", abs(m.steps("i2", "above", "i15") - before) < 1e-5)

# 4. retrieval by location on a clean 2-D grid, to places never related directly
rng = random.Random(3)
_, facts, qs = mb.grid_episode(5, 0.0, rng)
g = CognitiveMap(dim=2, robust=0.3)
for a, rel, b in facts:
    g.observe(str(a), rel, str(b))
g.sleep(300)
acc = np.mean([g.at(str(X), [("east", dx), ("north", dy)], top=1)[0][0] == str(ans) for X, dx, dy, ans in qs])
check("clean 2-D grid: every multi-step place found by location", acc == 1.0, f"{acc:.3f}")
check("different relations get (nearly) orthogonal directions",
      abs(float(g.rel["east"] @ g.rel["north"])) / (g.rel["east"].norm() * g.rel["north"].norm()) < 0.2)

# 5. the distance effect: with mislabelled facts, the map's errors fall on near pairs, chaining's on far ones
cfg = dict(lr=0.6, rel_lr=0.0, replay=40, robust=0.3, sleep=300, tol=0.3)
map_slopes, chain_slopes = [], []
for s in range(12):
    r = random.Random(100 + s)
    _, facts, qs = mb.transitive_episode(24, 0.1, r)
    map_slopes.append(mb.slope(mb.by_distance(mb.map_transitive(facts, qs, cfg, s), qs)))
    chain_slopes.append(mb.slope(mb.by_distance(mb.chain_transitive(facts, qs, 24, r), qs)))
check("distance effect: map's accuracy slope above chaining's", np.mean(map_slopes) > np.mean(chain_slopes),
      f"map {np.mean(map_slopes):+.4f} vs chain {np.mean(chain_slopes):+.4f}")

# 6. nothing is trained by gradient
check("no tensor requires a gradient", not m.pos.requires_grad and not any(v.requires_grad for v in m.rel.values()))

print("\nALL PASS" if ok else "\nFAILURES ABOVE")
sys.exit(0 if ok else 1)

In [ ]:
%%writefile /kaggle/working/neuromoe/PREREGISTRATION_convo.md
# Conversational reasoning: pre-registration

Committed before the reported run. Claim under test: a brain whose turn loop *is* the mechanisms - sparse lexical
codes, prefrontal working memory, a hippocampal map, an episodic store and a learned gate - can hold a
conversation and answer questions about it that were never stated, with one exposure per fact and no gradients.

## The task

A dialogue, turn by turn. `N` people are ordered on one dimension. Only adjacent pairs are ever stated, in
shuffled order, one sentence each. Some sentences refer to the previous speaker's subject by pronoun ("she is
older than Dan"). Every third turn is a distractor that carries no fact. A fraction `noise` of the facts are
stated backwards. Then questions are asked, as further turns:

- **C1 inference** - pairs at distance >= 2 that were never stated together ("Is Alice older than Eve?")
- **C2 reference** - the same question with the subject replaced by a pronoun, immediately after a turn that
  makes the referent unambiguous
- **C3 span** - inference questions about facts stated at least 8 turns earlier, with distractors in between
- **C4 contradiction** - pairs whose chain contains a backwards fact

Answers are scored yes/no. "I don't know" is scored wrong, and also reported separately as abstention, because a
system that abstains honestly is not the same as one that guesses.

## Systems

- **brain** - `brain/agent.py`, the whole loop, one exposure per fact, no training
- **ablations** - the same agent with one mechanism removed: no working memory (pronouns unresolved), no map
  (episodic recall only), no gate (fixed reflex routing), no sleep (no consolidation)
- **lexical** - retrieval: answer from the most word-overlapping earlier turn. Can answer stated pairs, cannot
  infer.
- **chain** - working-memory bindings only, answered by following links (the memory-without-a-map control that
  the pseudospatial run used)
- **transformer**, **gru** - trained by backprop on thousands of generated dialogues of the same kind, then
  answering in context

## Protocol

Gate settings (vigilance, trust) are chosen on validation seeds 2000-2007 only. The map keeps the settings
already tuned in the pseudospatial run - no re-tuning. Reported episodes use seeds 0-9. Paired over seeds,
two-sided t-tests, Holm-Bonferroni across the four primary tests, alpha = 0.05.

## Primary tests

| id | comparison (N = 10, noise = 0.1) | metric |
|---|---|---|
| P1 | brain - lexical | C1 inference accuracy |
| P2 | brain - chain | C1 inference accuracy |
| P3 | brain - trained transformer | C1 inference accuracy |
| P4 | brain - brain without working memory | C2 reference accuracy |

## Predictions and decision rules

- P1 > 0 and P2 > 0: inference over what was said, not retrieval of it, and not link-following.
- P4 > 0: pronoun resolution comes from the bound slot, not from word order.
- P3 is open. If a trained transformer matches or beats the brain, the claim narrows to "the same competence
  from one exposure per fact, with no training".
- If P1 and P2 are not significant, the conversational claim is not supported.

## Exploratory

C3 and C4; abstention rates; the gate's learning curve (share of turns selected by an expert rather than a
reflex); every other N and noise level; footprint against the trained baselines' parameter counts.

In [ ]:
import subprocess, sys
for test in ("tests/check_tissue.py", "tests/check_brain.py", "tests/check_chat.py", "tests/check_cogmap.py"):
    r = subprocess.run([sys.executable, f"/kaggle/working/neuromoe/{test}"], capture_output=True, text=True, cwd="/kaggle/working/neuromoe")
    print(r.stdout[-3000:], r.stderr[-1500:])
    assert r.returncode == 0, f"{test} failed - not running the benchmark"

In [ ]:
import subprocess, sys
# two seeds and a short training run first: a bug should cost two minutes, not the whole session
r = subprocess.run([sys.executable, "-u", "/kaggle/working/neuromoe/convo_bench.py", "--seeds", "2", "--steps", "200",
                    "--n", "10", "--noise", "0.1", "--out", "/kaggle/working/results/smoke"],
                   capture_output=True, text=True, cwd="/kaggle/working/neuromoe")
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, "smoke run failed - not starting the real one"
print("SMOKE OK")

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-u", "/kaggle/working/neuromoe/convo_bench.py", "--seeds", "10", "--steps", "20000",
                    "--n", "10", "--noise", "0.1", "--out", "/kaggle/working/results/convo"],
                   capture_output=True, text=True, cwd="/kaggle/working/neuromoe")
print(r.stdout[-20000:], r.stderr[-4000:])

In [ ]:
import subprocess, sys
# exploratory: does the margin hold as the dialogue gets longer and noisier?
for n, noise in ((6, 0.0), (10, 0.0), (14, 0.1), (10, 0.2)):
    r = subprocess.run([sys.executable, "-u", "/kaggle/working/neuromoe/convo_bench.py", "--seeds", "5", "--steps", "20000",
                        "--n", str(n), "--noise", str(noise),
                        "--out", f"/kaggle/working/results/convo_n{n}_noise{noise}"],
                       capture_output=True, text=True, cwd="/kaggle/working/neuromoe")
    print(f"=== N={n} noise={noise} ===")
    print(r.stdout[-3000:], r.stderr[-1500:])

Verdicts: `results/convo/ANALYSIS_convo.md`. Every episode's numbers are in the `convo_*.json` beside it, and the size sweep is in the `convo_n*` folders.